# 🧬 Nesprin-2 SR56-A isolated helix32 minibinder — LARGE256 + refine

目标为 canonical **6646–6677 的原 32 aa 孤立片段**：`MQCQDFHQLSQDLLLWLATAESRRQKAHVTSP`。生成、MPNN、RF3 和 refine 使用同一片段；binder **40–60 aa**。整个片段可作结合表面，无固定热点，不添加折叠态或受力状态门控。

正式默认：**8×32 骨架（lowmem=False）、每骨架 12 序列、RF3 每条 5 模型；前 10 个骨架设计 × 3 轮 refine**。主 RF3 序列验证上限 3,072；refine 上限 4,480；总计 7,552 条。实际数量及各阶段分母完整记账。

只排除无效结构数据；接触、碰撞、置信度和 RMSD 只报告。保留所有模型，展示和 refine 按有效输出中的原生 ranking_score 选优。预测输出不是实验结合结果。

保留原 36 单元、顺序和 ID；所有输出为空。输入档案只读，运行产物写入独立目录。文件/CPU 检查与真实 GPU 推理分别核验，不自动启动远程任务。

本文件内置已核验 model_3、原 job request 与 canonical FASTA 的最小输入包，默认解包到本分支准备目录；无需本机归档路径。


精简版：六份入口与共享 resources.zip；生成结果直接写入六个阶段目录。旧运行用旧代码继续，新版本不迁移旧检查点。


# 🧬 交互式蛋白质设计管道 

基于 **[rc-foundry](https://github.com/RosettaCommons/foundry）** 仓库。

| 步骤 | 模型 | 功能 |
|------|------|------|
| **1** | RFD3 | 固定模体主链，扩散生成新骨架 |
| **2** | MPNN | 为骨架设计氨基酸序列 |
| **3** | RF3 | 从序列预测 3D 结构，验证可折叠性 |
| **4** | — | RF3 ranking_score 排名 + RMSD 验证 → 导出前 5 |

### pipline
```
输入: PDB 
  │
  ▼  RFD3   固定模体 → 扩散生成骨架 
  │
  ▼  MPNN   为每个骨架设计 12 条序列
  │
  ▼  RF3    完整折叠环境重折叠验证 → ranking_score / pLDDT / pTM 报告
  │
  ▼  排名   RF3 ranking_score 排序 + RMSD 辅助参考 → 导出前 5
```

> **兼容性**：本地 Jupyter / JupyterLab。参数集中在 🎛️ Cell 中修改。


---
## 第 0 步：环境检查与导入

每次打开 Notebook 先运行此 Cell，完成 GPU 检测、检查点定位和依赖导入。

In [ ]:
# 【可修改设置】启动与环境设置：包路径、输出位置、恢复开关、普通依赖和已有权重路径。
# SR56_FEEDBACK_BEGIN
import sys as _feedback_sys, os as _feedback_os, importlib.util as _feedback_importlib
from pathlib import Path as _feedback_Path
import hashlib as _feedback_hashlib

def _feedback_safe(operation):
    try:
        return operation()
    except Exception as error:
        print('[SR56 feedback disabled operation] ' + type(error).__name__ + ': ' + str(error), file=_feedback_sys.stderr)

class _FeedbackNull:
    def event(self, *args, **kwargs): pass
    def configure(self, *args, **kwargs): pass
    def finish(self, *args, **kwargs): pass

_standalone_previous_feedback = globals().get("_feedback")

# 直接运行设置：通常留空即可；从任意工作目录运行时填写完整包目录。
# 包的绝对路径；留空时从当前目录/父目录定位。移动包或从其他工作目录启动时可填写。
STANDALONE_PACKAGE_ROOT = ""
# 留空：每次生成新运行目录。恢复：填写原目录，并设下面的 RESUME=True。
# 本次独立运行根目录；留空自动新建。恢复时填写原目录；禁止用新版本接管旧版本断点。
STANDALONE_OUTPUT_ROOT = ""
# 实验短名；留空为未命名实验。仅影响文件夹名称，不改变科学参数。
RUN_LABEL = ""
# False 创建新运行；True 恢复上面指定、身份匹配的运行；不会复制历史候选到新运行。
STANDALONE_RESUME = False
# 固定资源身份，由发布程序生成；不要手动修改或用它绕过资源完整性检查。
RESOURCE_BUNDLE_SHA256 = '1b5de3447aa151aaac2b74511f1b7cc7cb25813ebc369e0a60c86839c3c2a3a0'

def _prepare_resources(package):
    import json, zipfile, stat, tempfile, shutil, fcntl, io
    archive=package/'resources.zip'
    if archive.is_symlink() or not archive.is_file():
        raise RuntimeError('缺少发布根 resources.zip；请保留完整运行包')
    if archive.stat().st_size>64*1024**2: raise RuntimeError('资源 ZIP 超出大小上限')
    raw=archive.read_bytes()
    if _feedback_hashlib.sha256(raw).hexdigest()!=RESOURCE_BUNDLE_SHA256:
        raise RuntimeError('resources.zip 散列不符；拒绝加载其他版本或损坏资源')
    cache=package/'.resources'
    if cache.is_symlink(): raise RuntimeError('资源缓存不能是符号链接')
    cache.mkdir(exist_ok=True)
    dest=cache/RESOURCE_BUNDLE_SHA256
    lockpath=cache/'.prepare.lock'
    if lockpath.is_symlink(): raise RuntimeError('资源缓存锁不能是符号链接')
    with lockpath.open('a+') as lock:
        fcntl.flock(lock.fileno(),fcntl.LOCK_EX)
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            infos=z.infolist(); names=[i.filename for i in infos]
            if len(infos)>128 or len(names)!=len(set(names)) or sum(i.file_size for i in infos)>64*1024**2:
                raise RuntimeError('资源包成员重复或超出大小上限')
            for i in infos:
                p=_feedback_Path(i.filename)
                if (p.is_absolute() or '..' in p.parts or '\\' in i.filename or ':' in i.filename
                        or not p.parts or any(part in ('','.') for part in i.filename.split('/'))
                        or i.is_dir() or stat.S_ISLNK(i.external_attr>>16)
                        or i.flag_bits&1 or i.file_size>16*1024**2):
                    raise RuntimeError('资源包含不安全成员：'+i.filename)
            if 'RESOURCE_MANIFEST.json' not in names:
                raise RuntimeError('资源包缺少清单')
            manifest=json.loads(z.read('RESOURCE_MANIFEST.json'))
            if manifest.get('schema')!='sr56-compact-resources-v1':
                raise RuntimeError('不兼容的资源清单')
            files=manifest['files']; byname={item['path']:item for item in files}
            if len(byname)!=len(files) or set(names)!=(set(byname)|{'RESOURCE_MANIFEST.json'}):
                raise RuntimeError('资源包成员与清单不一致')
            def check(root):
                if root.is_symlink() or not root.is_dir(): raise RuntimeError('资源缓存无效')
                for name,item in byname.items():
                    p=root/name
                    if (any(q.is_symlink() for q in (p,*p.parents) if q!=root.parent)
                            or not p.is_file() or p.stat().st_size!=item['bytes']
                            or _feedback_hashlib.sha256(p.read_bytes()).hexdigest()!=item['sha256']):
                        raise RuntimeError('资源缓存缺失或被修改：'+name)
                if (root/'RESOURCE_MANIFEST.json').read_bytes()!=z.read('RESOURCE_MANIFEST.json'):
                    raise RuntimeError('资源缓存清单被修改')
            if dest.exists() or dest.is_symlink():
                check(dest)
            else:
                temp=_feedback_Path(tempfile.mkdtemp(prefix='.prepare-',dir=cache))
                try:
                    for i in infos:
                        p=temp/i.filename;p.parent.mkdir(parents=True,exist_ok=True)
                        p.write_bytes(z.read(i))
                    check(temp)
                    _feedback_os.replace(temp,dest)
                finally:
                    if temp.exists(): shutil.rmtree(temp)
    return dest


def _feedback_boot():
    global _feedback, _standalone_runtime, _feedback_paths, _feedback_assessed
    if (_standalone_previous_feedback is not None
            and getattr(_standalone_previous_feedback, '_standalone_lock', None)
            and not _standalone_previous_feedback.finished):
        _refusal = RuntimeError('此 kernel 已有运行会话；不要重复启动单元。继续后续单元，或中断并重启 kernel 后显式恢复。')
        _refusal.sr56_setup_rejected = True
        raise _refusal
    if _standalone_previous_feedback is not None and getattr(_standalone_previous_feedback, 'finished', False):
        _standalone_previous_feedback.close()
    cwd = _feedback_Path.cwd()
    candidates = ([_feedback_Path(STANDALONE_PACKAGE_ROOT).expanduser()] if STANDALONE_PACKAGE_ROOT
                  else [cwd, cwd.parent])
    roots = [p.resolve() for p in candidates if (p/'resources.zip').is_file()
             and (p/'SR56_A_minibinder/SR56_A_minibinder.ipynb').is_file()]
    roots = list(dict.fromkeys(roots))
    if len(roots) != 1:
        raise RuntimeError('请将工作目录设为本包目录；也可在本单元填写 STANDALONE_PACKAGE_ROOT 绝对路径。')
    package = roots[0]
    resource_root = _prepare_resources(package)
    # Do not import an installed or another cached runtime with the same name.
    runtime_file = resource_root/'standalone_runtime.py'
    runtime_spec = _feedback_importlib.spec_from_file_location('sr56_standalone_runtime', runtime_file)
    _standalone_runtime = _feedback_importlib.module_from_spec(runtime_spec)
    _feedback_sys.modules['sr56_standalone_runtime'] = _standalone_runtime
    exec(compile(runtime_file.read_bytes(), str(runtime_file), 'exec'), _standalone_runtime.__dict__)
    notebook = package/'SR56_A_minibinder/SR56_A_minibinder.ipynb'
    editor_path = globals().get('__vsc_ipynb_file__')
    if editor_path and _feedback_Path(editor_path).resolve() != notebook.resolve():
        raise RuntimeError('VS Code 当前 notebook 与所选包入口不一致；请打开包内对应入口。')
    _feedback = _standalone_runtime.start(package=package, notebook=notebook,
        branch='A_minibinder', namespace='sr56_a_helix32_minibinder_large256_refine_v3_20261002_chain_terminal_oxt_v1_20261004_output_v1_20261007_compact_v2_20261007_layout_v3', output=STANDALONE_OUTPUT_ROOT,
        resume=STANDALONE_RESUME, resource_root=resource_root, run_label=RUN_LABEL)
    _feedback_paths = {}
    _feedback_assessed = {}
    print('独立运行模式：', 'A_minibinder')
    print('运行 ID：', _feedback.run['run_id'], '执行批次：', _feedback.attempt)
    print('运行目录：', _feedback.root)
    print('无需工作台服务；使用当前 notebook kernel 的科研环境。')

_feedback_boot()

def _feedback_id(di, si, mi, rnd=0, pi=None, k=None):
    return f'main:D{di}:S{si}:M{mi}' if rnd == 0 else f'refine:D{di}:C{rnd}:P{pi}:K{k}:S{si}:M{mi}'

def _feedback_model(rec, mi, out, check, di, si, generated, mpnn, rnd=0, pi=None, k=None, parent_path=None):
    row = next(r for r in rec['models'] if r['mi'] == mi)
    model_id = _feedback_id(di, si, mi, rnd, pi, k)
    pair_id = model_id.rsplit(':M', 1)[0]
    path = parent_path if rnd == 0 else f'{parent_path}-C{rnd}P{pi}K{k}S{si}M{mi}'
    _feedback.event('MODEL', 'main_rf3' if rnd == 0 else 'refine_rf3', key=model_id,
        id=model_id, pair_id=pair_id, design_index=di, sequence_index=si, model_index=mi,
        round=rnd, parent_index=pi, backbone=k, path=path, parent_path=parent_path,
        parent_id=_feedback_paths.get((di, parent_path)), sequence=rec['input_seq'],
        valid_output=check['valid_output'], errors=check['errors'], metrics=check['metrics'],
        summary=out.summary_confidences,
        generated_path=str(_feedback_Path(generated).resolve()), mpnn_path=str(_feedback_Path(mpnn).resolve()),
        cif=str(_feedback_Path(row['cif']).resolve()), conf=str(_feedback_Path(row['conf']).resolve()),
        assessment_artifacts={'cif_sha256':_sha256_file(row['cif']), 'conf_sha256':_sha256_file(row['conf'])})
    _feedback_assessed.setdefault(pair_id, {})[mi] = {'mi':mi, 'valid_output':check['valid_output'], 'errors':check['errors'], 'metrics':check['metrics'], 'summary':out.summary_confidences}
    if rnd:
        _feedback_paths[(di, path)] = model_id

def _feedback_pair(rec, selected, di, si, generated, mpnn, rnd=0, pi=None, k=None, parent_path=None):
    pair_id = _feedback_id(di, si, 0, rnd, pi, k).rsplit(':M', 1)[0]
    _feedback.event('PAIR', 'main_rf3' if rnd == 0 else 'refine_rf3', key=pair_id,
        id=pair_id, pair_id=pair_id, design_index=di, sequence_index=si, round=rnd,
        parent_index=pi, backbone=k, path=parent_path, input_seq=rec['input_seq'],
        models=rec['models'], n_requested=rec['n_requested_models'], n_generated=len(rec['models']),
        generated_path=str(_feedback_Path(generated).resolve()), mpnn_path=str(_feedback_Path(mpnn).resolve()),
        selected_mi=selected[0] if selected else None, assessed_models=[_feedback_assessed[pair_id][r['mi']] for r in rec['models'] if r['mi'] in _feedback_assessed.get(pair_id, {})])

def _feedback_baseline(di, si, mi, path):
    _feedback_paths[(di, path)] = _feedback_id(di, si, mi)

def _feedback_beam(di, rnd, topk):
    for candidate in topk:
        c = candidate
        child_id = _feedback_id(di, c['si'], c['mi'], rnd, c['pi'], c['k'])
        _feedback.event('LINEAGE', 'refine_round', key=child_id, child_id=child_id,
            parent_id=_feedback_paths.get((di, c['parent_path'])), round=rnd, design_index=di,
            selected=True, path=f'{c["parent_path"]}-C{rnd}P{c["pi"]}K{c["k"]}S{c["si"]}M{c["mi"]}')
# SR56_FEEDBACK_END
# 最后修改: 2026-08-05 06:54
import sys, os, time, tempfile, json
import urllib.request
from pathlib import Path
# ── 依赖检查（可选安装；默认不联网、不升级既有 CUDA 环境）──
# False 仅检查现有环境；True 仅安装缺失的普通依赖。不升级 Torch/CUDA/Foundry，不下载权重。
INSTALL_MISSING = False   # True: 仅补齐缺失普通依赖（复用现有环境，不升级/不重装）
import importlib.util as _ilu
_DEP_SPECS = (('numpy', 'numpy'), ('torch', 'torch'), ('lightning', 'lightning'),
              ('biotite', 'biotite'), ('atomworks', 'rc-foundry[all]'), ('foundry', 'rc-foundry[all]'))
_missing_deps = [m for m, _sp in _DEP_SPECS if _ilu.find_spec(m) is None]
if _missing_deps:
    if not INSTALL_MISSING:
        raise RuntimeError('缺少依赖 ' + str(_missing_deps) + '；如确需安装请设 INSTALL_MISSING=True 后重跑本节'
                           '（本流程不自动下载大体积模型权重）')
    import subprocess as _subp
    for _m, _spec in _DEP_SPECS:
        if _ilu.find_spec(_m) is None:
            print('  ⬇️ 安装缺失依赖: ' + _spec)
            _subp.check_call([sys.executable, '-m', 'pip', 'install', _spec])

import numpy as np
import torch
from lightning.fabric import seed_everything

# ── 导入依赖 ──
from atomworks.io.utils.visualize import view
from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
from atomworks.io.utils.io_utils import to_cif_file
from biotite.structure import get_residue_starts, superimpose
from biotite.sequence import ProteinSequence

# ═══════════ 自包含 helpers（原布局修正版）═══════════
# 原 cell 依赖 debug/pipeline_fixes.py(重建版) 与已丢失的 debug/resume.py。
# 本块内联两者被调用的等义实现，保持 _fix / _resume 调用点不变；不 exec 大段字符串。
import re as _re, hashlib as _hashlib, csv as _csv
from types import SimpleNamespace
from biotite.structure.io.pdbx import CIFFile as _CIFFile, get_structure as _get_structure

_HOTSPOT_RE = _re.compile(r"^([A-Za-z])(\d+)-(\d+)$")

def _res_starts(aa):
    return get_residue_starts(aa, add_exclusive_stop=True)

def _chain_list(aa):
    """[(chain_id_str, [该链残基的全局残基序])]，按链出现顺序。"""
    st = _res_starts(aa)
    if len(st) < 2:
        return []
    ch = _chain_strs(aa.chain_id[st[:-1]])
    out, i = [], 0
    while i < len(st) - 1:
        j = i
        while j + 1 < len(st) - 1 and ch[j + 1] == ch[i]:
            j += 1
        out.append((str(ch[i]), list(range(i, j + 1))))
        i = j + 1
    return out

def _chain_res_order(aa, chain):
    st = _res_starts(aa)
    if len(st) < 2:
        return []
    ch = _chain_strs(aa.chain_id[st[:-1]])
    return [i for i in range(len(st) - 1) if ch[i] == str(chain)]

def _chain_seq(aa, chain):
    st = _res_starts(aa)
    if len(st) < 2:
        return ""
    return "".join(ProteinSequence.convert_letter_3to1(aa.res_name[st[i]])
                   for i in _chain_res_order(aa, chain))

def _input_target_seq():
    _inp = globals().get('input_aa'); _tc = globals().get('target_chain')
    if _inp is None or not _tc:
        return None
    _s = _chain_seq(_inp, _tc)
    return _s or None

def _resolve_target_chain(aa):
    """按『选定靶标链完整序列精确相等』定位靶标链；唯一命中才返回，否则 None（不猜、不取子串）。"""
    exp = _input_target_seq()
    if exp is None:
        return None
    hits = [c for c, _ in _chain_list(aa) if _chain_seq(aa, c) == exp]
    return hits[0] if len(hits) == 1 else None

def _complex_components(aa, eid):
    """完整复合物 → RF3 纯序列 components（无坐标、无模板）。"""
    comps = []
    for c, _ in _chain_list(aa):
        seq = _chain_seq(aa, c)
        if not seq:
            raise RuntimeError(f"链 {c} 序列为空，无法构建 RF3 序列输入")
        comps.append({"chain_id": str(c), "seq": seq, "chain_type": "polypeptide(L)"})
    if not comps:
        raise RuntimeError("结构无链，无法构建 RF3 序列输入")
    return {"name": str(eid), "components": comps}

def _backbone_coord_list(aa, chain=None, res_positions=None):
    """按 (链序, 残基序, N/CA/C/O) 严格顺序返回坐标 (n,3)。
    每个残基 N/CA/C/O 必须恰好各 1 个且坐标有限，否则返回 None（拒绝跳过/取首个/截断）。"""
    st = _res_starts(aa)
    if len(st) < 2:
        return None
    ch = _chain_strs(aa.chain_id[st[:-1]])
    out = []
    for i in range(len(st) - 1):
        if chain is not None and ch[i] != str(chain):
            continue
        if res_positions is not None and i not in res_positions:
            continue
        blk = aa[st[i]:st[i + 1]]
        for an in ("N", "CA", "C", "O"):
            sel = blk.atom_name == an
            if int(sel.sum()) != 1:
                return None
            xyz = np.asarray(blk.coord[sel][0], dtype=float)
            if not np.all(np.isfinite(xyz)):
                return None
            out.append(xyz)
    return np.asarray(out, dtype=float) if out else None

def _backbone_residue_count(aa, chain):
    return len(_chain_res_order(aa, chain))

def _fit_transform(P, Q):
    """Kabsch：求 proper rotation R,t 使 R@P+t≈Q（无镜像）。形状必须一致且 >=3，秩>=2，否则 None。"""
    P = np.asarray(P, dtype=float); Q = np.asarray(Q, dtype=float)
    if P.shape != Q.shape or P.ndim != 2 or P.shape[0] < 3 or P.shape[1] != 3:
        return None
    if not (np.all(np.isfinite(P)) and np.all(np.isfinite(Q))):
        return None
    Pc = P.mean(0); Qc = Q.mean(0)
    if np.linalg.matrix_rank(P - Pc) < 2 or np.linalg.matrix_rank(Q - Qc) < 2:
        return None
    H = (P - Pc).T @ (Q - Qc)
    U, S, Vt = np.linalg.svd(H)
    d = float(np.sign(np.linalg.det(Vt.T @ U.T)))
    D = np.diag([1.0, 1.0, d])
    R = Vt.T @ D @ U.T
    return R, (Qc - R @ Pc)

def _aligned_rmsd(ref_xyz, mov_xyz):
    """严格等形叠合 RMSD（不 min 截断）。"""
    if ref_xyz is None or mov_xyz is None:
        return float('nan')
    P = np.asarray(mov_xyz, float); Q = np.asarray(ref_xyz, float)
    rt = _fit_transform(P, Q)
    if rt is None:
        return float('nan')
    R, t = rt
    d = (P @ R.T + t) - Q
    return float(np.sqrt(np.mean(np.sum(d * d, axis=1))))

def _paired_binder_chains(aa_ref, aa_mov):
    """几何配对 binder 链：两侧必须恰好为『1 靶标 + 1 binder』且 binder 残基数相等。
    不做序列相等要求（MPNN/refine 会重设计 binder 序列）；不满足 → []（不可评估，不猜）。"""
    tgt_r = _resolve_target_chain(aa_ref); tgt_m = _resolve_target_chain(aa_mov)
    if tgt_r is None:
        return []
    if tgt_m is None:
        return []
    ref_b = [c for c, _ in _chain_list(aa_ref) if c != tgt_r]
    mov_b = [c for c, _ in _chain_list(aa_mov) if c != tgt_m]
    if len(ref_b) != 1 or len(mov_b) != 1:
        return []
    if _backbone_residue_count(aa_ref, ref_b[0]) != _backbone_residue_count(aa_mov, mov_b[0]):
        return []
    return [(ref_b[0], mov_b[0])]

def _binder_seq_of(aa):
    """binder 链 1-letter 序列（用于独立 MPNN 序列一致性核对）。"""
    _bd = _extract_binder(aa)
    _st = get_residue_starts(_bd)
    return ''.join(ProteinSequence.convert_letter_3to1(rn) for rn in _bd.res_name[_st])

def _binder_backbone_rmsd(aa_ref, aa_mov):
    """binder 结合姿态 RMSD（共享几何）：先在靶标链主链上求整体刚体变换，同一变换作用于 binder，
    再测 binder 与参考同原子的主链偏差。自拟合会掩盖结合姿态漂移（回归：位移 20Å 必须报 20Å）。
    仅接受几何上的『1 靶标 + 1 binder』且原子数一致；否则 NaN（不回退自拟合、不 min 截断）。"""
    if globals().get('design_task') != 'binder':
        return _aligned_rmsd(_backbone_coord_list(aa_ref), _backbone_coord_list(aa_mov))
    tgt_r = _resolve_target_chain(aa_ref); tgt_m = _resolve_target_chain(aa_mov)
    if tgt_r is None or tgt_m is None:
        return float('nan')
    Q = _backbone_coord_list(aa_ref, tgt_r); P = _backbone_coord_list(aa_mov, tgt_m)
    if Q is None or P is None or Q.shape != P.shape:
        return float('nan')
    rt = _fit_transform(P, Q)
    if rt is None:
        return float('nan')
    R, t = rt
    num, den = 0.0, 0
    for rc, mc in _paired_binder_chains(aa_ref, aa_mov):
        Qb = _backbone_coord_list(aa_ref, rc); Pb = _backbone_coord_list(aa_mov, mc)
        if Qb is None or Pb is None or Qb.shape != Pb.shape:
            return float('nan')          # 缺残基/缺原子 → 拒绝，不截断
        d = (Pb @ R.T + t) - Qb
        num += float(np.sum(d * d)); den += len(Qb)
    if den == 0:
        return float('nan')
    return float(np.sqrt(num / den))

def _binder_self_rmsd(aa_ref, aa_mov):
    """binder 自洽重折叠 RMSD（诊断，不用于门控）：每条 binder 链独立拟合后的主链 RMSD。"""
    if globals().get('design_task') != 'binder':
        return _aligned_rmsd(_backbone_coord_list(aa_ref), _backbone_coord_list(aa_mov))
    num, den = 0.0, 0
    for rc, mc in _paired_binder_chains(aa_ref, aa_mov):
        Qb = _backbone_coord_list(aa_ref, rc); Pb = _backbone_coord_list(aa_mov, mc)
        if Qb is None or Pb is None or Qb.shape != Pb.shape:
            return float('nan')
        rt = _fit_transform(Pb, Qb)
        if rt is None:
            return float('nan')
        R, t = rt
        d = (Pb @ R.T + t) - Qb
        num += float(np.sum(d * d)); den += len(Qb)
    if den == 0:
        return float('nan')
    return float(np.sqrt(num / den))

def _target_frame_transform(aa_ref, aa_mov):
    tgt_r = _resolve_target_chain(aa_ref); tgt_m = _resolve_target_chain(aa_mov)
    if tgt_r is None or tgt_m is None:
        return None
    Q = _backbone_coord_list(aa_ref, tgt_r); P = _backbone_coord_list(aa_mov, tgt_m)
    if Q is None or P is None or Q.shape != P.shape:
        return None
    return _fit_transform(P, Q)

def _transform_coords(aa, rt):
    if rt is None:
        return aa
    R, t = rt
    out = aa.copy()
    out.coord = aa.coord @ R.T + t
    return out

def _ca_dev_map(aa_ref, aa_mov, exclude_chain=None):
    """严格 (chain, resi) 对应 Cα 偏差；键缺失/数量不一致 → None（不截断）。"""
    def _m(aa):
        st = _res_starts(aa); ch = _chain_strs(aa.chain_id[st[:-1]])
        d = {}
        for i in range(len(st) - 1):
            if exclude_chain is not None and ch[i] == str(exclude_chain):
                continue
            blk = aa[st[i]:st[i + 1]]; sel = blk.atom_name == "CA"
            if bool(sel.any()):
                d[(str(ch[i]), int(aa.res_id[st[i]]))] = np.asarray(blk.coord[sel][0], float)
        return d
    a = _m(aa_ref); b = _m(aa_mov)
    if len(a) != len(b) or any(k not in b for k in a):
        return None
    return {k: float(np.linalg.norm(a[k] - b[k])) for k in a}

def _hotspot_local_positions(chain, s, e, input_aa=None):
    """input_aa 热点段 → 链内残基序位置；请求残基缺失则 raise（fail closed）。
    仅允许映射到选定靶标链（拒绝把其他链的热点编号映射到靶标）。"""
    _inp = input_aa if input_aa is not None else globals().get('input_aa')
    if _inp is None:
        raise RuntimeError("hotspot 定位需要 input_aa")
    _tc = globals().get('target_chain')
    if _tc and str(chain) != str(_tc):
        raise RuntimeError(f"热点链 {chain} != 选定靶标链 {_tc}；拒绝跨链映射")
    order = _chain_res_order(_inp, chain)
    if not order:
        raise RuntimeError(f"热点链 {chain} 不在输入结构中")
    _st = _res_starts(_inp)
    ids = [int(_inp.res_id[_st[i]]) for i in order]   # 修复：i 是残基序号，必须经 starts 取 res_id
    loc = [k for k, rid in enumerate(ids) if s <= rid <= e]
    want = e - s + 1
    if len(loc) != want:
        raise RuntimeError(f"热点段 {chain}{s}-{e} 在输入结构中缺失残基（期望 {want}，实际 {len(loc)}）")
    return loc

def _parse_hotspots(text):
    out = []
    for part in str(text or "").split(","):
        part = part.strip()
        if not part:
            continue
        m = __import__('re').match(r"^([A-Za-z])(\d+)-(\d+)$", part)   # 自包含：不依赖模块级 _HOTSPOT_RE
        if not m:
            raise ValueError(f"热点段格式非法: {part!r}（应为 链+起-止，如 A798-815）")
        a, b = int(m.group(2)), int(m.group(3))
        if a > b:
            raise ValueError(f"热点段起>止: {part!r}")
        out.append((m.group(1), a, b))
    if not out:
        raise ValueError("hotspot_residues 为空，拒绝无门控运行")
    return out

def _hotspot_segment_rmsd(aa_ref, aa_mov, chain, s, e, input_aa=None):
    try:
        loc = _hotspot_local_positions(chain, s, e, input_aa)
    except Exception:
        return float('nan')
    tgt_r = _resolve_target_chain(aa_ref); tgt_m = _resolve_target_chain(aa_mov)
    if tgt_r is None or tgt_m is None:
        return float('nan')
    ord_r = _chain_res_order(aa_ref, tgt_r); ord_m = _chain_res_order(aa_mov, tgt_m)
    if len(ord_r) <= max(loc) or len(ord_m) <= max(loc):
        return float('nan')
    Q = _backbone_coord_list(aa_ref, tgt_r, set(ord_r[k] for k in loc))
    P = _backbone_coord_list(aa_mov, tgt_m, set(ord_m[k] for k in loc))
    if Q is None or P is None or Q.shape != P.shape or Q.shape[0] < 3:
        return float('nan')
    return _aligned_rmsd(Q, P)

def _hotspot_rmsd_map(aa_ref, aa_mov, input_aa=None, hotspot_residues=None):
    """逐热点段主链 RMSD（精确位置映射，无子串匹配）。返回 {seg: rmsd|nan}。"""
    _inp = input_aa if input_aa is not None else globals().get('input_aa')
    _hs = hotspot_residues if hotspot_residues is not None else globals().get('hotspot_residues')
    out = {}
    for chain, s, e in _parse_hotspots(_hs):
        out[f"{chain}{s}-{e}"] = _hotspot_segment_rmsd(aa_ref, aa_mov, chain, s, e, _inp)
    return out

def _motif_rmsd(aa_ref, aa_mov):
    """非 binder：按输入编号精确位置映射的模体主链 RMSD。"""
    _inp = globals().get('input_aa')
    if _inp is None:
        return float('nan')
    _mc = globals().get('motif_chain'); _ms = globals().get('motif_start'); _me = globals().get('motif_end')
    try:
        loc = _hotspot_local_positions(_mc, int(_ms), int(_me), _inp)
    except Exception:
        return float('nan')
    tgt_r = _resolve_target_chain(aa_ref); tgt_m = _resolve_target_chain(aa_mov)
    if tgt_r is None or tgt_m is None:
        return float('nan')
    ord_r = _chain_res_order(aa_ref, tgt_r); ord_m = _chain_res_order(aa_mov, tgt_m)
    if len(ord_r) <= max(loc) or len(ord_m) <= max(loc):
        return float('nan')
    Q = _backbone_coord_list(aa_ref, tgt_r, set(ord_r[k] for k in loc))
    P = _backbone_coord_list(aa_mov, tgt_m, set(ord_m[k] for k in loc))
    if Q is None or P is None or Q.shape != P.shape or Q.shape[0] < 3:
        return float('nan')
    return _aligned_rmsd(Q, P)

def _anchor_contacts_from_complex(refolded_aa, input_aa, hotspot_residues, cutoff=4.5):
    """full traversal 接触统计（分块遍历全部原子，无抽样/无 min 截断）。"""
    out = {}
    try:
        tgt = _resolve_target_chain(refolded_aa)
        if tgt is None:
            return out
        binder = [c for c, _ in _chain_list(refolded_aa) if c != tgt]
        if not binder:
            return out
        st = _res_starts(refolded_aa)
        ch = _chain_strs(refolded_aa.chain_id[st[:-1]])
        bmask = np.zeros(len(refolded_aa), dtype=bool)
        for i in range(len(st) - 1):
            if ch[i] in binder:
                bmask[st[i]:st[i + 1]] = True
        bcoords = np.asarray(refolded_aa.coord[bmask], float)
        ord_t = _chain_res_order(refolded_aa, tgt)
        for chain, s, e in _parse_hotspots(hotspot_residues):
            try:
                loc = _hotspot_local_positions(chain, s, e, input_aa)
            except Exception:
                continue
            tmask = np.zeros(len(refolded_aa), dtype=bool)
            for k in loc:
                if k < len(ord_t):
                    i = ord_t[k]; tmask[st[i]:st[i + 1]] = True
            tcoords = np.asarray(refolded_aa.coord[tmask], float)
            if len(tcoords) == 0 or len(bcoords) == 0:
                continue
            mind = float('inf'); nc = 0
            step = max(1, len(bcoords) // 200)
            for i0 in range(0, len(bcoords), step):
                d2 = np.sum((bcoords[i0:i0 + step, None, :] - tcoords[None, :, :]) ** 2, axis=2)
                mind = min(mind, float(np.sqrt(d2.min())))
                nc += int((d2.min(axis=1) < cutoff * cutoff).sum())
            out[f"{chain}{s}-{e}"] = {"n_contacts": nc, "min_dist_aa": mind,
                                      "n_target_atoms": int(len(tcoords))}
    except Exception:
        return {}
    return out

def _renumber_residues(aa):
    """R1：同残基所有原子共享 res_id，重编号 1..N（不改原对象）。"""
    sub = aa.copy()
    st = get_residue_starts(sub, add_exclusive_stop=True)
    lens = np.diff(st)
    sub.res_id = np.repeat(np.arange(1, len(st)), lens).astype(aa.res_id.dtype)
    return sub

def _format_rmsd(value, width=8):
    if not isinstance(value, (int, float, np.floating, np.integer)):
        return "N/A".rjust(width)
    v = float(value)
    if np.isnan(v) or np.isinf(v):
        return "N/A".rjust(width)
    return f"{v:.3f}".rjust(width)

def _prepare_atom_array_for_mpnn(atom_array):
    sub = atom_array.copy()
    if "res_name" in sub.get_annotation_categories() and len(sub) > 0:
        sub = sub[~np.isin(sub.res_name, ["HOH", "WAT"])]
    if "atom_id" in sub.get_annotation_categories():
        sub.del_annotation("atom_id")
    sub.chain_id = _chain_strs(sub.chain_id)
    return sub

def _refine_mpnn_input_dict(atom_array, input_aa, batch_size=5):
    d = {"batch_size": int(batch_size), "remove_waters": True}
    tgt = _resolve_target_chain(atom_array)
    if not tgt:
        raise RuntimeError("refine MPNN 无法唯一定位靶标链，拒绝全链设计")
    d["fixed_chains"] = [str(tgt)]
    return d

def _build_partial_diffusion_spec(atom_array, partial_t, fixed_chains=None,
                                  redesign_sequences=False, is_non_loopy=True):
    tmp_dir = Path(tempfile.mkdtemp(prefix="pdiff_", dir=str(PREP_DIR)))
    tmp_cif = tmp_dir / "input.cif"; tmp_json = tmp_dir / "spec.json"
    to_cif_file(atom_array, str(tmp_cif))
    fixed_set = [str(c) for c in (fixed_chains or [])]
    fixed_atoms, unfixed_seq = {}, {}
    st = get_residue_starts(atom_array)
    ch = _chain_strs(atom_array.chain_id[st])
    for c in sorted(set(ch)):
        idx = np.where(ch == c)[0]
        lo, hi = int(atom_array.res_id[st[idx]].min()), int(atom_array.res_id[st[idx]].max())
        sel = f"{c}{lo}-{hi}"
        if c in fixed_set:
            fixed_atoms[sel] = "BKBN"
        elif redesign_sequences:
            unfixed_seq[sel] = True
    spec = {"input": str(tmp_cif.resolve()), "partial_t": float(partial_t),
            "is_non_loopy": bool(is_non_loopy)}
    if fixed_atoms:
        spec["select_fixed_atoms"] = fixed_atoms
    if unfixed_seq:
        spec["select_unfixed_sequence"] = unfixed_seq
    with open(tmp_json, "w") as f:
        json.dump({"design_spec": spec}, f)
    return spec, str(tmp_cif), str(tmp_json)


_fix = SimpleNamespace(
    renumber_residues=_renumber_residues,
    detect_target_chain=lambda aa, input_aa=None: _resolve_target_chain(aa),
    hotspot_rmsd=_hotspot_rmsd_map,
    anchor_contacts_from_complex=_anchor_contacts_from_complex,
    build_partial_diffusion_spec=_build_partial_diffusion_spec,
    prepare_atom_array_for_mpnn=_prepare_atom_array_for_mpnn,
    refine_mpnn_input_dict=_refine_mpnn_input_dict,
    format_rmsd=_format_rmsd,
)

# ── 断点续跑（JSON + 工件哈希；不信任旧 pickle）──
def _sha256_file(path, required=True):
    """内容 sha256（每次实算，不做秒级缓存）；失败时 required=True 抛错（不得静默 None）。"""
    try:
        h = _hashlib.sha256()
        with open(str(path), 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 20), b''):
                h.update(chunk)
        return h.hexdigest()
    except Exception as e:
        if required:
            raise RuntimeError(f"无法读取/哈希文件（拒绝静默通过）: {path} ({e})") from e
        return None

_WEIGHT_SHA_MEMO = {}

def _weight_sha_memo(path):
    """权重完整内容 sha256，按 (path,size,mtime_ns,ctime_ns) 记忆化（文件变则重算）。"""
    stt = os.stat(str(path))
    key = (str(path), stt.st_size, stt.st_mtime_ns, stt.st_ctime_ns)
    if key not in _WEIGHT_SHA_MEMO:
        _WEIGHT_SHA_MEMO[key] = _sha256_file(path)
    return _WEIGHT_SHA_MEMO[key]

def _weight_identity(path):
    """权重身份：size+mtime_ns+ctime_ns + 完整内容 sha256（记忆化，可核验）；stat 不可用 → raise。"""
    try:
        stt = os.stat(str(path))
    except Exception as e:
        raise RuntimeError(f"权重文件不可用: {path} ({e})") from e
    return {"size": stt.st_size, "mtime_ns": stt.st_mtime_ns, "ctime_ns": stt.st_ctime_ns,
            "sha256": _weight_sha_memo(path)}

def _engine_sources():
    """已安装引擎来源/版本（断点配置可核验：换环境/换权重来源会改变身份）。"""
    _out = {}
    for _m in ('torch', 'atomworks', 'rfd3', 'rf3', 'mpnn'):
        try:
            _mod = __import__(_m)
            _out[_m] = {'file': str(getattr(_mod, '__file__', '')),
                        'version': str(getattr(_mod, '__version__', '') or '')}
            _root = Path(getattr(_mod, '__file__', '')).parent
            _entries = {'rfd3': ('engine.py', 'inference/input_parsing.py', 'model/inference_sampler.py'),
                        'rf3': ('inference_engines/rf3.py', 'utils/inference.py'),
                        'mpnn': ('inference_engines/mpnn.py',)}.get(_m, ())
            _out[_m]['source_sha256'] = {p: _sha256_file(_root / p) for p in _entries if (_root / p).is_file()}
        except Exception as _e:
            _out[_m] = {'error': type(_e).__name__}
    return _out

def _config_fingerprint(cfg):
    return _hashlib.sha256(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()

def _read_cif(path):
    return _get_structure(_CIFFile.read(str(path)), model=1)

def _save_stage(ckpt_path, stage, payload, files=()):
    p = Path(ckpt_path); p.parent.mkdir(parents=True, exist_ok=True)
    data = {}
    if p.exists() and p.stat().st_size > 0:
        try:
            data = json.loads(p.read_text()) or {}
        except Exception as e:
            raise RuntimeError(f"断点文件损坏，拒绝复用: {p} ({e})") from e
    for _existing_rec in data.values(): _check_checkpoint_identity(_existing_rec)
    rec = {"cfg": payload.get("cfg"), "payload": dict(payload),
           "files": [{"path": str(f), "sha256": _sha256_file(f)} for f in files],
           "ts": time.time()}
    rec["schema"] = CACHE_SCHEMA
    rec["run_namespace"] = _RUN_NS
    data[str(stage)] = rec
    tmp = p.with_suffix(p.suffix + ".tmp")
    tmp.write_text(json.dumps(_json_value(data), ensure_ascii=False, indent=1, allow_nan=False))
    os.replace(tmp, p)

def _load_stage(ckpt_path, stage, missing='error'):
    p = Path(ckpt_path)
    if not p.exists():
        return None
    try:
        data = json.loads(p.read_text())
    except Exception as e:
        raise RuntimeError(f"断点文件损坏，拒绝复用: {p} ({e})") from e
    for _entry in (data or {}).values(): _check_checkpoint_identity(_entry)
    rec = (data or {}).get(str(stage))
    if not rec:
        return None
    _check_checkpoint_identity(rec)
    for f in rec.get("files", []):
        fp = f.get("path")
        if not fp:
            raise RuntimeError(f"断点工件路径无效，拒绝复用 {stage}")
        if not Path(fp).exists():
            if missing == 'restart':
                print(f"  ↪ {stage} 工件缺失，仅重新执行该阶段: {fp}")
                return None
            raise RuntimeError(f"断点工件缺失，拒绝复用 {stage}: {fp}")
        if f.get("sha256") and _sha256_file(fp) != f["sha256"]:
            raise RuntimeError(f"断点工件被修改，拒绝复用 {stage}: {fp}")
    return rec.get("payload")

def _reconstruct_rfd3_designs(rfd3_dir, pdb_tag, n_batches, diffusion_batch_size):
    pat = _re.compile(rf"^rfd3_{_re.escape(str(pdb_tag))}_B(\d+)_D(\d+)\.cif$")
    got = {}
    for p in Path(rfd3_dir).glob(f"rfd3_{pdb_tag}_*.cif"):
        m = pat.match(p.name)
        if m:
            key = (int(m.group(1)), int(m.group(2)))
            if key in got:
                raise RuntimeError(f"RFD3 断点 B{key[0]}_D{key[1]} 重复，拒绝复用")
            got[key] = p
    exp_keys = {(b, i2) for b in range(int(n_batches)) for i2 in range(int(diffusion_batch_size))}
    if set(got) != exp_keys:
        _miss = sorted(exp_keys - set(got))[:5]; _extra = sorted(set(got) - exp_keys)[:5]
        raise RuntimeError(f"RFD3 断点 ID 网格不符（缺失 {_miss} / 多余 {_extra}），拒绝复用")
    out = []
    for (b, i2) in sorted(got):
        p = got[(b, i2)]
        aa = _read_cif(p)
        out.append((f"{b}_d{i2}", aa, len(set(zip([str(c) for c in aa.chain_id], aa.res_id)))))
    return out

def _reconstruct_mpnn_results(mpnn_dir, pdb_tag, designs_to_process, mpnn_seqs_per_backbone, accepted_indices=None):
    accepted_indices = set(range(len(designs_to_process))) if accepted_indices is None else set(accepted_indices)
    pat = _re.compile(rf"^mpnn_{_re.escape(str(pdb_tag))}_B(\d+)_D(\d+)_S(\d+)\.cif$")
    groups = {}
    for p in Path(mpnn_dir).glob(f"mpnn_{pdb_tag}_*_S*.cif"):
        m = pat.match(p.name)
        if m:
            groups.setdefault((int(m.group(1)), int(m.group(2))), {})[int(m.group(3))] = p
    # 设计名格式 "B_dD"（来自 RFD3 生成循环）→ 期望的 (B, D) 精确网格
    exp = {}
    for d_idx, (d_name, _da, _dnr) in enumerate(designs_to_process):
        try:
            _b, _dd = str(d_name).rsplit("_d", 1)
            exp[d_idx] = (int(_b), int(_dd))
        except Exception as e:
            raise RuntimeError(f"设计名 {d_name!r} 无法解析 B_dD 网格，拒绝复用") from e
    exp_keys = {exp[i] for i in accepted_indices}
    if set(groups) != exp_keys:
        _miss = sorted(exp_keys - set(groups))[:5]; _extra = sorted(set(groups) - exp_keys)[:5]
        raise RuntimeError(f"MPNN 断点 ID 网格不符（缺失 {_miss} / 多余 {_extra}），拒绝复用")
    out = []
    for d_idx, (d_name, d_aa, d_nr) in enumerate(designs_to_process):
        if d_idx not in accepted_indices:
            out.append((d_name, d_aa, []))
            continue
        g = groups[exp[d_idx]]
        if set(g) != set(range(int(mpnn_seqs_per_backbone))):
            raise RuntimeError(f"MPNN 断点 {exp[d_idx]} 的 S 索引网格不符：{sorted(g)}，拒绝复用")
        seqs = [SimpleNamespace(atom_array=_read_cif(g[s])) for s in range(int(mpnn_seqs_per_backbone))]
        out.append((d_name, d_aa, seqs))
    return out

def _rf3_progress_path(persist_dir):
    d = Path(persist_dir); d.mkdir(parents=True, exist_ok=True)
    return d / "progress.csv"

def _save_rf3_output(persist_dir, d_idx, s_idx, model, model_idx=0):
    d = Path(persist_dir); d.mkdir(parents=True, exist_ok=True)
    cifp = d / f"rf3_{int(d_idx)}_{int(s_idx)}_m{int(model_idx)}.cif"
    confp = d / f"rf3_{int(d_idx)}_{int(s_idx)}_m{int(model_idx)}_conf.json"
    to_cif_file(model.atom_array, str(cifp))
    conf = model.confidences or {}
    conf_json = {}
    for k, v in conf.items():
        try:
            conf_json[k] = v.tolist() if hasattr(v, 'tolist') else v
        except Exception:
            conf_json[k] = None
    _write_json_atomic(confp, {'summary': dict(model.summary_confidences or {}), 'confidences': conf_json})
    return str(cifp), str(confp)

def _append_rf3_progress(persist_dir, d_idx, s_idx, summary, cifp, confp, seq_head, model_idx=0):
    csvp = _rf3_progress_path(persist_dir)
    new = not csvp.exists()
    with open(csvp, "a", newline="") as f:
        w = _csv.writer(f)
        if new:
            w.writerow(["d_idx", "s_idx", "model_idx", "ranking", "plddt", "ptm", "iptm",
                        "pae", "clash", "cifp", "confp", "cif_sha", "conf_sha", "seq_head"])
        w.writerow([int(d_idx), int(s_idx), int(model_idx),
                    summary.get('ranking_score'), summary.get('overall_plddt'),
                    summary.get('ptm'), summary.get('iptm'), summary.get('overall_pae'),
                    summary.get('has_clash'), cifp, confp,
                    _sha256_file(cifp), _sha256_file(confp), str(seq_head)])

def _reconstruct_rf3_results(csv_path, persist_dir):
    """返回该 (d,s) 的全部已持久化模型（不预选 ranking）；工件哈希校验失败/损坏一律 raise。
    记录形如 (d_idx, s_idx, model_idx, seq_head, summary, output)；QC 与选优由调用方（与首次运行一致）。"""
    csvp = Path(csv_path)
    if not csvp.exists():
        return []
    try:
        rows = list(_csv.DictReader(csvp.open()))
    except Exception as e:
        raise RuntimeError(f"RF3 断点进度 CSV 损坏，拒绝复用: {csvp} ({e})") from e
    out = []
    for r in rows:
        try:
            d = int(r["d_idx"]); s = int(r["s_idx"]); mi = int(r.get("model_idx") or 0)
        except Exception as e:
            raise RuntimeError(f"RF3 断点行 ID 非法，拒绝复用: {r} ({e})") from e
        cifp = r.get("cifp"); confp = r.get("confp")
        if not cifp or not Path(cifp).exists():
            raise RuntimeError(f"RF3 断点结构工件缺失，拒绝复用: {cifp}")
        if r.get("cif_sha") and _sha256_file(cifp) != r["cif_sha"]:
            raise RuntimeError(f"RF3 断点结构工件被修改，拒绝复用: {cifp}")
        if not confp or not Path(confp).exists():
            raise RuntimeError(f"RF3 断点置信度工件缺失，拒绝复用: {confp}")
        if r.get("conf_sha") and _sha256_file(confp) != r["conf_sha"]:
            raise RuntimeError(f"RF3 断点置信度工件被修改，拒绝复用: {confp}")
        try:
            aa = _read_cif(cifp)
        except Exception as e:
            raise RuntimeError(f"RF3 断点结构无法解析，拒绝复用: {cifp} ({e})") from e
        try:
            conf = json.loads(Path(confp).read_text())
        except Exception as e:
            raise RuntimeError(f"RF3 断点置信度 JSON 损坏，拒绝复用: {confp} ({e})") from e
        out.append((d, s, mi, r.get("seq_head", ""), conf.get("summary", {}),
                    SimpleNamespace(atom_array=aa, confidences=conf.get("confidences", {}),
                                    summary_confidences=conf.get("summary", {}))))
    return out

_resume = SimpleNamespace(
    config_fingerprint=_config_fingerprint,
    save_stage=_save_stage,
    load_stage=_load_stage,
    reconstruct_rfd3_designs=_reconstruct_rfd3_designs,
    reconstruct_mpnn_results=_reconstruct_mpnn_results,
    rf3_progress_path=_rf3_progress_path,
    save_rf3_output=_save_rf3_output,
    append_rf3_progress=_append_rf3_progress,
    reconstruct_rf3_results=_reconstruct_rf3_results,
)


_cfg_fp = lambda: _resume.config_fingerprint(_current_cfg())
print("  🔧 自包含 helpers 已就绪（_fix/_resume 内联；无 debug 目录依赖）")

print("  🔧 debug 修复已集成: S2/M1/L7 库补丁 + 管线修复函数")

# ── 环境变量 ──

# ── torch 兼容补丁（仅旧版 PyTorch 无 device_type 参数时启用）──
import torch as _torch
try:
    _torch.is_autocast_enabled(device_type='cuda')
except TypeError:
    _orig = _torch.is_autocast_enabled
    def _patched(*_args, **_kwargs): return _orig()   # 吞参无参转发（兼容旧 torch is_autocast_enabled 无参签名）
    _torch.is_autocast_enabled = _patched
    print("  ⚠️ 已应用 torch.is_autocast_enabled 兼容补丁（旧版 PyTorch）")

# ── 静默第三方库刷屏日志（MACE 缓存 / out_dir / extra_fields 等）──
import logging, warnings
try:
    from foundry.utils.logging import CachedDataFilter   # foundry 官方的 MACE 缓存过滤器
except ImportError:
    CachedDataFilter = None

class _WarnOnlyFilter(logging.Filter):
    """过滤所有 WARNING + 特定刷屏 INFO 消息；保留其余 INFO/DEBUG/ERROR/CRITICAL。"""
    _NOISE_INFO = ("structures to predict", "Predicting structure")   # rf3 引擎每条推理刷屏的 INFO
    def filter(self, record):
        if record.levelno == logging.WARNING:
            # 不丢弃：转存到 warnings.log（保留真实引擎告警，避免刷屏 stdout）
            try:
                _wpath = globals().get('OUT_DIR') or os.getcwd()
                with open(os.path.join(str(_wpath), "warnings.log"), "a") as _wf:
                    _wf.write(f"[{time.strftime('%H:%M:%S')}] {record.name}: {record.getMessage()}\n")
            except Exception:
                pass
            return False
        if record.levelno == logging.INFO:
            return not any(k in record.getMessage() for k in self._NOISE_INFO)
        return True

_QUIET_FILTER = _WarnOnlyFilter()

def _chain_strs(chain_ids):
    """biotite chain_id → str 数组（兼容 bytes / np.bytes_ / unicode / np.str_ 类型差异）"""
    return np.array([str(c) for c in chain_ids], dtype=str)

def _extract_binder(aa):
    """binder 模式：按靶标链完整序列精确匹配剥离靶标，binder 链重编号 1-N。
    靶标无法唯一确定且存在多链 → raise（拒绝静默按链序/子串剥离）。"""
    if globals().get('design_task') != "binder":
        return aa
    tgt = _resolve_target_chain(aa)
    if tgt is None:
        _chs = [c for c, _ in _chain_list(aa)]
        if len(_chs) <= 1:
            return aa
        raise RuntimeError(f"binder 模式无法在多链结构 {_chs} 中唯一定位靶标链"
                           f"（需 input_aa/target_chain 完整序列精确匹配），拒绝静默剥离。")
    st = _res_starts(aa); ch = _chain_strs(aa.chain_id[st[:-1]])
    mask = np.zeros(len(aa), dtype=bool)
    for i in range(len(st) - 1):
        if ch[i] != tgt:
            mask[st[i]:st[i + 1]] = True
    if not mask.any():
        return aa
    return _renumber_residues(aa[mask])

def _quiet_logging():
    """过滤所有 WARNING，保留 INFO 进度 / ERROR 错误。
    关键：logging 的 filter 不继承（level 才继承），必须遍历所有已注册 logger 逐个挂载；
    每次 import 新模块（如 Cell 16/23 的 rf3 引擎）后需重新调用以覆盖新 logger。"""
    for _name in list(logging.Logger.manager.loggerDict):
        if _name.startswith(("atomworks", "rfd3", "rf3", "mace")):
            _lg = logging.getLogger(_name)
            if _QUIET_FILTER not in _lg.filters:
                _lg.addFilter(_QUIET_FILTER)
    # 保留 INFO 级别（引擎进度信息可见）
    logging.getLogger().setLevel(logging.INFO)
    # Python warnings 通道全忽略（extra_fields 等走 warnings 的也一并静默）
    warnings.filterwarnings("ignore")

_quiet_logging()

# ── 断点续跑（修正版：统一 JSON + 工件哈希，不使用 pickle）──
# 初始化的派生检查点位置；后续目录准备单元统一写到 state/checkpoint.json，无需修改。
CKPT_FILE = None  # 由参数 Cell 设置路径（Cell 8）

# ── 记录设备；只有存在待推理步骤时才要求 CUDA ──
# 从当前环境自动检测计算设备；不是模拟 GPU 成功的开关，不需手改。
DEVICE_TYPE = 'cuda' if torch.cuda.is_available() else 'unavailable'
print('运行设备:', DEVICE_TYPE, '；CPU 可完成输入/空结果处理，不执行模型推理')

# ── 定位检查点 ──
# 已有模型权重目录：先读取已有环境设置，再检查默认目录；本包不含权重。
CKPT_DIR = Path(
    os.environ.get("FOUNDRY_CHECKPOINT_DIRS", "").split(":")[0]
    or next((p for p in (Path.cwd() / "models", Path.cwd().parent / "models") if p.is_dir()),
            Path.home() / ".foundry" / "checkpoints")
)
if not CKPT_DIR.exists():
    CKPT_DIR = Path.home() / ".foundry" / "checkpoints"
print(f"检查点目录: {CKPT_DIR}")

# 检查点文件名支持环境变量覆盖（默认值适用于 foundry install）
# RFD3 已有权重文件；环境变量 RFD3_CKPT 优先，默认使用 CKPT_DIR 下对应文件。
RFD3_CKPT = os.environ.get("RFD3_CKPT", str(CKPT_DIR / "rfd3_latest.ckpt"))
# ProteinMPNN 已有权重文件；环境变量 MPNN_CKPT 优先。
MPNN_CKPT = os.environ.get("MPNN_CKPT", str(CKPT_DIR / "proteinmpnn_v_48_020.pt"))
# RF3 已有权重文件；环境变量 RF3_CKPT 优先。
RF3_CKPT  = os.environ.get("RF3_CKPT", str(CKPT_DIR / "rf3_foundry_01_24_latest_remapped.ckpt"))

for name, path in [("RFD3", RFD3_CKPT), ("MPNN", MPNN_CKPT), ("RF3", RF3_CKPT)]:
    if Path(path).exists():
        print(f"  ✓ {name}: {Path(path).name}")
    else:
        print(f"  ✗ {name}: 缺失！请运行 foundry install rfd3 proteinmpnn rf3")

print(f"\n✅ 环境就绪")
from IPython.display import display   # ═══ 预览显示修复（view→display）═══
from IPython.display import HTML as _HTML
_CLC_CSS = """<style>
/* ═══ 折叠报表：跟随 VSCode 主题自动切换（浅/深色）═══ */
.clc{ --clc-border:#d0d7de; --clc-th:#f0f3f6; --clc-tr:#f8fafc; --clc-pre:#f6f8fa;
  --clc-txt:#1f2328; --clc-mut:#57606a; --clc-link:#0969da; --clc-sum-bg:#eef1f4; }
details.clc { margin:4px 0; }
details.clc > summary { cursor:pointer; font-weight:600; color:var(--clc-txt);
  background:var(--clc-sum-bg); padding:2px 8px; border-radius:4px; }
details.clc table { border-collapse:collapse; font-size:11px; font-family:monospace;
  border:1px solid var(--clc-border); color:var(--clc-txt); background:transparent; }
details.clc th { border:1px solid var(--clc-border); padding:3px 6px; background:var(--clc-th); color:var(--clc-txt); }
details.clc td  { border:1px solid var(--clc-border); padding:3px 6px; color:var(--clc-txt); }
details.clc tr.anchor td { background:var(--clc-tr); }
details.clc pre { max-height:420px; overflow:auto; font-size:11px; line-height:1.5;
  padding:8px 10px; border-radius:6px; white-space:pre-wrap;
  background:var(--clc-pre); color:var(--clc-txt); }
details.clc small { color:var(--clc-mut); }
details.clc a { color:var(--clc-link); }
@media (prefers-color-scheme: dark) {
  .clc { --clc-border:#3d444d; --clc-th:#21262d; --clc-tr:#161b22; --clc-pre:#161b22;
    --clc-txt:#e6edf3; --clc-mut:#8b949e; --clc-link:#58a6ff; --clc-sum-bg:#21262d; }
}
</style>"""

def _collapsible(title, body_html, open_state=False):
    """可折叠 HTML 块：多轮次/多条目输出默认收起，点击展开。配色跟随 VSCode 主题。"""
    _op = " open" if open_state else ""
    return _HTML(_CLC_CSS + f"<details{_op} class='clc' style='margin:4px 0'><summary>{title}</summary>{body_html}</details>")

def _fold_out(title, open_state=False):
    """捕获代码块的 stdout → 折叠为 <details><pre> 可展开块（批量信息默认收起）。配色跟随主题。"""
    import io as _io
    from contextlib import redirect_stdout as _redir
    class _Ctx:
        def __init__(self):
            self._buf = _io.StringIO()
        def __enter__(self):
            self._r = _redir(self._buf); self._r.__enter__(); return self._buf
        def __exit__(self, *exc):
            self._r.__exit__(*exc)
            _t = self._buf.getvalue().replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
            if _t.strip():
                display(_HTML(_CLC_CSS + f"<details{' open' if open_state else ''} class='clc' style='margin:4px 0'><summary>{title}</summary><pre style='max-height:420px;overflow:auto;font-size:11px;line-height:1.5;padding:8px 10px;border-radius:6px;white-space:pre-wrap'>{_t}</pre></details>"))
            return False
    return _Ctx()

# ═══ 3D 视图：3Dmol.js 本地内联（不依赖外部 CDN，离线/内网可用）═══
import pathlib as _pl, os as _os
_3DMOL_LIB = None
def _load_3dmol_lib():
    global _3DMOL_LIB
    if _3DMOL_LIB is not None:
        return _3DMOL_LIB
    _p = _feedback_Path(_standalone_runtime.__file__).parent/'assets/3Dmol-min.js'
    if _p.is_file():
        _3DMOL_LIB = _p.read_text(encoding='utf-8')
        return _3DMOL_LIB
    return None

_3DMOL_CDN = "loadScriptAsync('https://cdn.jsdelivr.net/npm/3dmol@2.5.3/build/3Dmol-min.js')"
def _lib3d():
    """3Dmol-min.js 库内容（本地内联，离线可用）。"""
    lib = _load_3dmol_lib()
    return lib or ''
def _view3d(v):
    """py3Dmol 视图 → HTML 片段（CDN→Promise.resolve；库由 _view3d_self/_wrap_3d 前置内联）。"""
    html = v._make_html()
    if _3DMOL_CDN in html and _lib3d():
        html = html.replace(_3DMOL_CDN, 'Promise.resolve()')
    return html
def _view3d_self(v):
    """单个视图 → 自包含 HTML：每个 3D 输出都内联 3Dmol 库一次，不依赖其他输出的全局 $3Dmol。"""
    lib = _lib3d()
    frag = _view3d(v)
    return ('<script>' + lib + '</script>' + frag) if lib else frag
def _wrap_3d(html):
    """整块 3D HTML（可含多个视图）→ 前置内联库一次，输出自包含。幂等：已含库则跳过。"""
    lib = _lib3d()
    if not lib or not html:
        return html
    s = str(html)
    if '/*! For license information please see 3Dmol-min.js' in s:
        return html
    return '<script>' + lib + '</script>' + s
# ═══════════ 官方/业界参考指标（按模式 × 流程） ═══════════
_REF_MODE_CN = {"monomer": "单体", "motif_scaffolding": "模体支架",
                "binder": "结合剂", "symmetry": "对称体", "ligand_design": "配体设计"}

def _ref_metrics(design_task):
    """当前模式各流程的官方/业界参考指标（RFD3 骨架 / MPNN 序列 / RF3 验证）。
    返回 {流程: [(指标名, 参考阈值, 官方/业界来源), ...]}。单体无 ipTM（无结合界面）。"""
    refs = {
        "RFD3": [
            ("靶标参考系姿态RMSD", "≤3.0Å 本轮结构门槛", "本轮预设筛选约束"),
            ("设计性(可重折叠序列占比)", "≥50%", "RFdiffusion 论文"),
            ("核心局部RMSD", "≤1.5Å 本轮结构门槛", "本轮预设筛选约束"),
        ],
        "MPNN": [
            ("序列恢复率", "52–59%", "ProteinMPNN 论文 (Dauparas 2022)"),
            ("平均 logP(序列评分)", "越高越好(接近0)", "ProteinMPNN 论文"),
        ],
        "RF3": [
            ("pLDDT", ">0.7 高置信 / >0.9 很高", "AlphaFold2 官方"),
            ("pTM", ">0.5 可折叠 / >0.8 高质量", "AlphaFold2 官方"),
            ("ipTM", "原始相对复合物置信度；仅报告，不用于排除", "AF-Multimer 官方"),
            ("PAE", "<5Å 高置信", "AlphaFold2 官方"),
            ("clash", "0 冲突", "结构质量校验"),
        ],
    }
    if design_task == "monomer":
        refs["RF3"] = [r for r in refs["RF3"] if r[0] != "ipTM"]   # 单体无结合界面
    return refs

def _ref_panel(design_task, stage, computed=None):
    """渲染「官方/业界参考指标对照」可折叠表格。
    computed: {指标名: (实际值str, 达标bool|None)}；None 的指标标 —。"""
    _rows = []
    for _name, _ref, _src in _ref_metrics(design_task)[stage]:
        if computed and _name in computed:
            _val, _ok = computed[_name]
            _mark = "✅" if _ok else ("❌" if _ok is False else "—")
        else:
            _val, _mark = "—", "—"
        _rows.append(f"<tr><td>{_name}</td><td>{_val} {_mark}</td><td>{_ref}</td><td>{_src}</td></tr>")
    return _collapsible(f"📏 {_REF_MODE_CN.get(design_task, design_task)} · {stage} 官方/业界参考指标对照",
        "<table border='1' style='border-collapse:collapse;font-size:11px;font-family:monospace'>"
        "<tr><th>指标</th><th>实际值</th><th>参考阈值</th><th>官方/业界来源</th></tr>"
        + "".join(_rows) + "</table>")






# SR56_DISPLAY_BEGIN
def _sr56_fmt(value):
    import math
    if value is None: return '—'
    if isinstance(value, (float, int)) and not isinstance(value, bool):
        return f'{value:.4f}' if math.isfinite(float(value)) else '—'
    return str(value)

def _sr56_plot_value(value):
    import math
    try:
        result = float(value)
        return result if math.isfinite(result) else None
    except (TypeError, ValueError):
        return None

def _sr56_emit(title, body, expanded=False):
    import html
    try:
        from IPython.display import HTML
        show = globals().get('display')
        if callable(show):
            show(globals()['_collapsible'](html.escape(str(title)), body, expanded))
    except Exception as exc:
        print(str(title) + '（显示不可用：' + type(exc).__name__ + '）')

def _sr56_table(title, rows, columns, full_csv=None, note='', expanded=False):
    import csv, html
    from pathlib import Path
    try:
        count = 0; preview = []; writer = None; stream = None
        try:
            if full_csv is not None:
                target = Path(full_csv); target.parent.mkdir(parents=True, exist_ok=True)
                stream = target.open('w', newline='', encoding='utf-8')
                writer = csv.DictWriter(stream, fieldnames=[k for k, label in columns], extrasaction='ignore')
                writer.writeheader()
            for row in rows:
                if writer is not None: writer.writerow(row)
                count += 1
                if len(preview) < 50: preview.append(row)
        finally:
            if stream is not None: stream.close()
        esc = lambda value: html.escape(_sr56_fmt(value))
        head = ''.join('<th>' + esc(label) + '</th>' for key, label in columns)
        body = ''.join('<tr>' + ''.join('<td style="max-width:720px;overflow-wrap:anywhere;white-space:normal">' + esc(row.get(key)) + '</td>' for key, label in columns) + '</tr>' for row in preview)
        path = ('；完整 CSV：<code>' + esc(full_csv) + '</code>') if full_csv is not None else ''
        _sr56_emit(title, '<p>' + esc(note) + '</p><p>共 ' + str(count) + ' 行；仅显示前 ' + str(len(preview)) + ' 行' + path + '</p><div style="overflow:auto;max-height:520px"><table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table></div>', expanded)
    except Exception as exc:
        print(str(title) + '（显示表不可用：' + type(exc).__name__ + '）')

def _sr56_tick(stage, reused, requested, returned, index=None, total=None, reset=False):
    import time
    state = globals().setdefault('_sr56_display_counts', {})
    if reset or stage not in state: state[stage] = {'start':time.time(), 'steps':0, 'reused':0, 'requested':0, 'returned':0}
    row = state[stage]
    if reset: return
    row['steps'] += 1; row['reused'] += int(bool(reused)); row['requested'] += int(requested); row['returned'] += int(returned)
    if stage == 'RFD3' or row['steps'] <= 3 or row['steps'] % 50 == 0 or (total is not None and index == total):
        print(f"{stage} {index if index is not None else row['steps']}/{total if total is not None else '—'}；本步请求 {requested} / 返回 {returned} / 复用 {bool(reused)}；累计请求 {row['requested']} / 返回 {row['returned']}；断点复用 {row['reused']}；本次单元耗时 {time.time()-row['start']:.1f}s")

def _sr56_local_3dmol():
    from pathlib import Path
    runtime = globals().get('_standalone_runtime')
    filename = getattr(runtime, '__file__', '')
    if filename:
        path = Path(filename).parent/'assets/3Dmol-min.js'
        if path.is_file(): return path.read_text(encoding='utf-8')
    loader = globals().get('_lib3d')
    return loader() if callable(loader) else ''

def _sr56_wrap_local(fragment):
    import re
    library = _sr56_local_3dmol()
    if not library: return '<p>本地 3Dmol 库不可用；结构文件和表格仍可查看。</p>'
    fragment = re.sub(r"loadScriptAsync\(['\"][^'\"]*3[Dd]mol[^'\"]*['\"]\)", 'Promise.resolve()', str(fragment))
    return '<script>' + library + '</script>' + fragment

def _sr56_pdb_text(aa):
    try:
        import numpy as np
        from biotite.structure.io.pdb import PDBFile
        copy = aa.copy()
        # Rendering copy only: undefined coordinates cannot be drawn or serialized to PDB.
        # Do not impute coordinates or touch the original atom array / CIF exporter.
        copy = copy[np.isfinite(copy.coord).all(axis=1)]
        if len(copy) == 0: return ''
        if 'b_factor' in copy.get_annotation_categories():
            copy.b_factor = np.clip(np.nan_to_num(copy.b_factor, nan=0.0, posinf=0.0, neginf=0.0), 0, 999.99)
        file = PDBFile(); file.set_structure(copy)
        return '\n'.join(file.lines)
    except Exception:
        return ''

def _sr56_main_rows():
    for rank, item in enumerate(globals().get('rmsd_report', []), 1):
        di,name,summary,fragment,pose,si = item
        candidate = globals().get('design_cand', {}).get(di, {})
        mpnn = candidate.get('mpnn_aa'); out = candidate.get('rf3_out')
        sequence = _binder_seq_of(mpnn) if mpnn is not None else ''
        yield {'rank':rank, 'design_index':di, 'sequence_index':si, 'model_index':candidate.get('model_index'), 'sequence':sequence,
               **summary, **candidate.get('metrics', {})}

def _sr56_current_best():
    """Cell25 reads the current RF3 stage, before Cell27 builds the design ranking."""
    try:
        candidates = globals().get('all_rf3_results', [])
        if not candidates: return None
        name,di,si,short_sequence,summary,out = candidates[0]
        mi = globals()['_best_model_indices'][(di,si)]
        backbone = globals()['designs_to_process'][di][1]
        mpnn = globals()['all_mpnn_results'][di][2][si].atom_array
        sequence = _binder_seq_of(mpnn)
        metrics = _assess_model(mpnn,out.atom_array,summary)['metrics']
        table = {'rank':1,'design_index':di,'sequence_index':si,'model_index':mi,'sequence':sequence,**summary,**metrics}
        view_row = {'design_index':di, 'label':f'当前 RF3 最佳 D{di}:S{si}:M{mi}', 'sequence':sequence,
                    'summary':summary,'metrics':metrics,'rfd3':backbone,'mpnn':mpnn,'rf3':out.atom_array,'rf3_confidences':getattr(out,'confidences',{})}
        return {'table':table,'view':view_row}
    except Exception as exc:
        print('当前 RF3 最佳显示不可用：' + type(exc).__name__)
        return None

def _sr56_main_views(limit=10):
    rows = []
    for rank, item in enumerate(globals().get('rmsd_report', [])[:limit], 1):
        di,name,summary,fragment,pose,si = item; candidate = globals().get('design_cand', {}).get(di, {})
        out = candidate.get('rf3_out')
        if out is None: continue
        rows.append({'design_index':di,'label':f"#{rank} D{di}:S{si}:M{candidate.get('model_index')}",
                     'sequence':_binder_seq_of(candidate['mpnn_aa']), 'summary':summary, 'metrics':candidate.get('metrics', {}),
                     'rfd3':candidate['orig_aa'], 'mpnn':candidate['mpnn_aa'], 'rf3':out.atom_array,'rf3_confidences':getattr(out,'confidences',{})})
    return rows

def _sr56_render_info(aa, stage, confidences=None):
    import numpy as np
    info={'pdb':_sr56_pdb_text(aa),'target':None,'confidence':{},'binder_ca_plddt':None}
    try:
        info['target']=_resolve_target_chain(aa)
        if stage!='rf3' or info['target'] is None: return info
        conf=(confidences or {}).get('atom_plddts',[])
        chains=(confidences or {}).get('atom_chain_ids',[])
        if len(conf)!=len(aa) or len(chains)!=len(aa): return info
        p=np.asarray(conf,dtype=float)
        if not np.isfinite(p).all() or np.any(p<0) or np.any(p>1): return info
        mapping={}
        for c,a in zip(chains,aa.chain_id):
            mapping.setdefault(str(c),set()).add(str(a))
        if any(len(v)!=1 for v in mapping.values()) or len({next(iter(v)) for v in mapping.values()})!=len(mapping): return info
        if 'b_factor' not in aa.get_annotation_categories(): return info
        b=np.asarray(aa.b_factor,dtype=float)
        if not np.isfinite(b).all() or not np.allclose(b,p,atol=.0051,rtol=0): return info
        ca=(aa.atom_name=='CA')&(aa.chain_id!=info['target'])
        if not np.any(ca): return info
        info['binder_ca_plddt']=float(np.mean(b[ca])*100)
        for i in np.flatnonzero(ca):
            info['confidence'][str(aa.chain_id[i])+':'+str(int(aa.res_id[i]))]=float(b[i]*100)
    except Exception:
        pass
    return info

def _sr56_three_panel(title, entries, expanded=False):
    """Two lazy views: stage-selectable current path and fixed main baseline."""
    import html,json,uuid
    try:
        library=_sr56_local_3dmol()
        if not library:
            _sr56_emit(title,'<p>本地 3Dmol 不可用；指标与原始结构导出仍可使用。</p>',expanded);return
        payload=[]
        fields=[('ranking_score','ranking score',''),('binder_ca_plddt','binder CA pLDDT',' /100'),('iptm','ipTM',''),
                ('binder_self_rmsd_A','binder self RMSD',' Å'),('binder_pose_rmsd_A','binder pose RMSD',' Å'),
                ('target_fragment_rmsd_A','target fragment RMSD',' Å'),('target_contact_residues','接触残基',''),
                ('contact_atom_pairs','接触原子对',''),('min_distance_A','最短距离',' Å'),
                ('has_clash','RF3 原生碰撞',''),('has_geometric_clash','几何碰撞','')]
        def projected(row):
            structures={key:_sr56_render_info(row[key],key,row.get('rf3_confidences') if key=='rf3' else None) for key in ('rfd3','mpnn','rf3')}
            values={**row.get('summary',{}),**row.get('metrics',{}),'binder_ca_plddt':structures['rf3']['binder_ca_plddt']}
            return {'label':row['label'],'sequence':row.get('sequence',''),'structures':structures,
                    'values':[{'label':label,'value':('未提供' if values.get(key) is None else _sr56_fmt(values[key]))+unit} for key,label,unit in fields]}
        for row in entries[:50]:
            current=projected(row);di=row.get('design_index');c=globals().get('design_cand',{}).get(di)
            baseline=None
            if c and c.get('rf3_out') is not None:
                out=c['rf3_out'];si=c.get('s_idx',c.get('sequence_index',globals().get('design_best',{}).get(di,(None,None,None,None,None))[4] if di in globals().get('design_best',{}) else None))
                baseline=projected({'label':f"baseline D{di}:S{si}:M{c.get('model_index')} · 主阶段最佳",
                    'sequence':_binder_seq_of(c['mpnn_aa']),'summary':out.summary_confidences,'metrics':c.get('metrics',{}),
                    'rfd3':c['orig_aa'],'mpnn':c['mpnn_aa'],'rf3':out.atom_array,'rf3_confidences':getattr(out,'confidences',{})})
            payload.append({'current':current,'baseline':baseline})
        if not payload: _sr56_emit(title,'<p>没有可展示的候选。</p>',expanded);return
        uid='sr56_'+uuid.uuid4().hex
        options=''.join('<option value="'+str(i)+'">'+html.escape(r['current']['label'])+'</option>' for i,r in enumerate(payload))
        panels=''.join('<section style="min-width:0"><h4>'+label+'</h4><p data-identity="'+key+'" style="overflow-wrap:anywhere"></p><div data-metrics="'+key+'" style="display:grid;grid-template-columns:repeat(auto-fit,minmax(140px,1fr));gap:6px;font-size:12px"></div><p data-seq="'+key+'" style="overflow-wrap:anywhere;font:12px monospace"></p><div id="'+uid+'_'+key+'" style="height:350px;position:relative;overflow:hidden"></div><a data-download="'+key+'" download>下载当前展示 PDB 副本</a></section>' for key,label in [('current','当前路径'),('baseline','对应 baseline · 固定 RF3')])
        body='<div id="'+uid+'"><label>候选 <select data-candidate>'+options+'</select></label> <label>当前阶段 <select data-stage><option value="rf3">RF3</option><option value="mpnn">MPNN</option><option value="rfd3">RFD3</option></select></label> <label><input type="checkbox" data-target checked> 靶标</label> <label><input type="checkbox" data-binder checked> binder</label> <button type="button" data-reset>重置视角</button><p>靶标：中性灰 · RF3 binder：<span style="color:#0053d6">≥90 蓝</span> / <span style="color:#159ec7">70–90 青</span> / <span style="color:#977300">50–70 黄</span> / <span style="color:#da6b00">&lt;50 橙</span>；无置信度：固定色。标度 0–100。</p><div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,360px),1fr));gap:16px">'+panels+'</div><small>原坐标展示，不自动叠合。RFD3／MPNN 的指标区仅参照标题注明的 RF3 模型；pLDDT 不借用。下载 PDB 是绘图副本，原始 CIF 保留在阶段目录。</small></div>'
        data=json.dumps(payload,ensure_ascii=False,allow_nan=False).replace('<','\\u003c')
        script=r'''<script>(function(){const root=document.getElementById('__ID__'),rows=__DATA__,views={};if(!root)return;let active=false;const webglAvailable=(()=>{try{const canvas=document.createElement('canvas'),gl=canvas.getContext('webgl2')||canvas.getContext('webgl');if(!gl)return false;gl.getExtension('WEBGL_lose_context')?.loseContext();return true;}catch(_){return false;}})();const esc=x=>String(x).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));function release(){for(const v of Object.values(views))try{v.clear();}catch(_){}for(const canvas of root.querySelectorAll('canvas'))try{const gl=canvas.getContext('webgl2')||canvas.getContext('webgl');gl?.getExtension('WEBGL_lose_context')?.loseContext();}catch(_){}for(const k of Object.keys(views))delete views[k];for(const k of ['current','baseline'])document.getElementById('__ID___'+k).replaceChildren();active=false;const box=root.closest('details');if(box)box.open=false;}function draw(){if(window._sr56ActiveViews&&window._sr56ActiveViews.root!==root)window._sr56ActiveViews.release();window._sr56ActiveViews={root,release};const entry=rows[+root.querySelector('[data-candidate]').value],stage=root.querySelector('[data-stage]').value;for(const key of ['current','baseline']){const row=entry[key],kind=key==='baseline'?'rf3':stage,el=document.getElementById('__ID___'+key);root.querySelector('[data-identity="'+key+'"]').textContent=row?row.label+' · 展示 '+kind.toUpperCase()+(kind!=='rf3'?' · 指标参照此路径 RF3':''):'Baseline 未提供';root.querySelector('[data-metrics="'+key+'"]').innerHTML=row?row.values.map(x=>'<div><small>'+esc(x.label)+'</small><br><b>'+esc(x.value)+'</b></div>').join(''):'';root.querySelector('[data-seq="'+key+'"]').textContent=row?row.sequence:'';const a=root.querySelector('[data-download="'+key+'"]');a.hidden=!row;if(!row){if(views[key])views[key].removeAllModels();continue}const s=row.structures[kind];a.href='data:chemical/x-pdb;charset=utf-8,'+encodeURIComponent(s.pdb);a.download=key+'_'+kind+'.pdb';if(!window.$3Dmol||!webglAvailable){el.textContent='WebGL/3Dmol 不可用，指标与结构下载保留';continue}try{const v=views[key]||(views[key]=$3Dmol.createViewer(el,{backgroundColor:'#142b2d'}));v.removeAllModels();v.addModel(s.pdb,'pdb');const showT=root.querySelector('[data-target]').checked,showB=root.querySelector('[data-binder]').checked;v.setStyle({},{cartoon:{color:'#90c9b5',hidden:!showB}});if(s.target)v.setStyle({chain:s.target},{cartoon:{color:'#aaaeb4',hidden:!showT}});if(kind==='rf3'&&showB)for(const [id,p] of Object.entries(s.confidence)){const parts=id.split(':');v.setStyle({chain:parts[0],resi:+parts[1]},{cartoon:{color:p>=90?'#0053d6':p>=70?'#65cbf3':p>=50?'#ffdb13':'#ff7d45'}})}v.zoomTo();v.resize();v.render();}catch(e){el.textContent='WebGL 不可用；请使用指标与下载。'}}active=true}function resize(){if(active)for(const v of Object.values(views)){v.resize();v.render()}}root.querySelectorAll('select,input').forEach(el=>el.addEventListener('change',draw));root.querySelector('[data-reset]').addEventListener('click',draw);window.addEventListener('resize',resize);if(typeof ResizeObserver==='function')new ResizeObserver(resize).observe(root);const d=root.closest('details');if(d){d.addEventListener('toggle',()=>{if(d.open)draw();else release()});if(d.open)draw()}else draw()})();</script>'''.replace('__ID__',uid).replace('__DATA__',data)
        _sr56_emit(title,'<script>'+library+'</script>'+body+script,expanded)
    except Exception as exc:
        print(str(title)+'（显示不可用：'+type(exc).__name__+': '+str(exc)[:120]+'）')

def _sr56_refine_rows():
    paths = globals().get('_feedback_paths', {})
    retained = globals().get('_sr56_display_retained', [])
    sequences = {(row['design_index'], row['path']):row.get('sequence','') for row in retained}
    for di, history in globals().get('refine_history', {}).items():
        best = globals().get('refine_best_round', {}).get(di)
        for rnd,pose,plddt,ptm,k,si,path,self_r,fragment in history:
            yield {'design_index':di,'row_type':'baseline' if rnd==0 else '每轮代表', 'round':rnd, 'path':path,
                   'candidate_id':paths.get((di,path),'—'), 'whole_loop_best':bool(best and best[0]==rnd and best[6]==path),
                   'sequence':(_binder_seq_of(globals()['design_cand'][di]['mpnn_aa']) if rnd==0 else sequences.get((di,path),'')),
                   'ranking_score':globals().get('refine_rankings',{}).get(di,{}).get(rnd),'iptm':globals().get('refine_iptms',{}).get(di,{}).get(rnd),
                   'overall_plddt':plddt,'ptm':ptm,'binder_self_rmsd_A':self_r,'binder_pose_rmsd_A':pose,'target_fragment_rmsd_A':fragment}
    for row in retained:
        best = globals().get('refine_best_round', {}).get(row['design_index'])
        yield {**row, 'row_type':'retained 路径', 'whole_loop_best':bool(best and best[0]==row['round'] and best[6]==row['path'])}
# SR56_DISPLAY_END


---
## 🧭 SR56-A LARGE256 minibinder Controller

Run All 在原单元位置完成：

1. 核验 AF3 model_3、Q6ZWQ0-1 canonical 序列及来源散列；
2. 提取 canonical 6646–6677 的同一 32 aa 孤立片段，保留原坐标；
3. 生成 40–60 aa binder，保存全部有效及无效输出的审计与阶段分母；
4. 按有效输出中的原生 ranking_score 排名，运行三轮 refine 并保留 baseline / 全循环最佳结果；
5. 使用本分支独立运行目录及断点；数据无效时跳过，运行错误停止并保留已完成阶段。

整个片段均可作为结合表面，无固定热点。接触、碰撞、置信度、二级结构和 RMSD 仅报告，不添加折叠态或受力状态门控。
输入通过 NESPRIN_AF3_ZIP / NESPRIN_CANONICAL_FASTA 指定；运行产物通过 NESPRIN_RUN_ROOT 写入独立目录。


In [ ]:
# 【固定流程 · 无需修改】核验输入来源与运行身份，准备同一孤立 helix 靶标。
# ═══════════════════════════════════════════════════════════════
# SR56-A ISOLATED HELIX32 MINIBINDER LARGE256 + REFINE
# ═══════════════════════════════════════════════════════════════
from pathlib import Path
import json, time

PROJECT_ROOT = Path(os.environ.get('NESPRIN_RUN_ROOT', str(Path.cwd() / 'sr56_a_helix32_minibinder_large256_refine_20261002'))).expanduser().resolve()
if any(p.name == '90_ARCHIVE' for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]):
    raise RuntimeError('运行目录不能位于 90_ARCHIVE；请指定独立 NESPRIN_RUN_ROOT')
PREP_DIR = PROJECT_ROOT / 'prep'
PROXY_DIR = PREP_DIR
MONITOR_DIR = PROJECT_ROOT / 'monitor'
REPORT_DIR = PROJECT_ROOT / 'reports'
MAIN_REPORT_DIR = REPORT_DIR / 'main'
MAIN_RANKING_DIR = PROJECT_ROOT / 'rankings/main'
REFINE_RANKING_DIR = PROJECT_ROOT / 'rankings/refine'
SVG_MAIN_DIR = PROJECT_ROOT / 'svg/main'
SVG_REFINE_DIR = PROJECT_ROOT / 'svg/refine'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
MAIN_RF3_CHECKPOINT_DIR = CHECKPOINT_DIR / 'main/rf3'
REFINE_RF3_CHECKPOINT_DIR = CHECKPOINT_DIR / 'refine/rf3'
for _d in (PREP_DIR, PROXY_DIR, MONITOR_DIR, REPORT_DIR, MAIN_REPORT_DIR, MAIN_RANKING_DIR, REFINE_RANKING_DIR, SVG_MAIN_DIR, SVG_REFINE_DIR, CHECKPOINT_DIR, MAIN_RF3_CHECKPOINT_DIR, REFINE_RF3_CHECKPOINT_DIR): _d.mkdir(parents=True, exist_ok=True)

# ── Portable, verified inputs; no machine-specific source path required ──
import hashlib
_INPUT_ORIGIN = {}
def _resolve_verified_input(env_name, member):
    path = Path(_standalone_runtime.__file__).parent/'inputs'/member
    manifest = json.loads((path.parent.parent/'RUNTIME_FILES.json').read_text())['files']
    expected = next(row['sha256'] for row in manifest if row['path']=='inputs/'+member)
    if path.is_symlink() or not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest()!=expected:
        raise RuntimeError('已核验的共享输入缺失或被修改：'+member)
    _INPUT_ORIGIN[member] = 'verified_resources_zip'
    return path
AF3_SR54_56_ZIP = _resolve_verified_input('NESPRIN_AF3_ZIP', 'fold_nesprin_sr54_sr56.zip')
CANONICAL_FASTA_PATH = _resolve_verified_input('NESPRIN_CANONICAL_FASTA', 'Q6ZWQ0.fasta')
_source_report = {'origins':_INPUT_ORIGIN, 'AF3_zip':str(AF3_SR54_56_ZIP),
                   'canonical_fasta':str(CANONICAL_FASTA_PATH), 'resource_bundle_sha256':RESOURCE_BUNDLE_SHA256}
_source_report_path = PREP_DIR/'input_sources_manifest.json'
_source_report_temp = _source_report_path.with_suffix('.tmp')
_source_report_temp.write_text(json.dumps(_source_report, ensure_ascii=False, indent=2))
os.replace(_source_report_temp, _source_report_path)
print('已定位核验输入:', _INPUT_ORIGIN)

TARGET_SPECIES = 'Mus musculus'
TARGET_GENE = 'Syne2'
TARGET_UNIPROT = 'Q6ZWQ0-1'
TARGET_CHAIN = 'A'
TARGET_NAME = 'SR56_A_HELIX32_MINIBINDER_LARGE256_REFINE'
TARGET_HELIX = 'SR56-A isolated helix32'
BINDER_CLASS = 'minibinder'
TARGET_FRAGMENT_PATH = PROXY_DIR / 'AF3_model3_canonical_6646_6677_helix32.cif'
TARGET_HOTSPOTS = None
MONITOR_STATE_PATH = MONITOR_DIR / 'monitor_state.json'

def shortpep_monitor_update(stage, **kwargs):
    state = {'updated_at_unix': time.time(), 'project': TARGET_NAME, 'stage': stage,
             'search_size_backbones': diffusion_batch_size * n_batches,
             'binder_length': [BINDER_MIN_AA, BINDER_MAX_AA]}
    state.update(kwargs)
    temp = MONITOR_STATE_PATH.with_suffix('.tmp')
    temp.write_text(json.dumps(state, ensure_ascii=False, indent=2))
    os.replace(temp, MONITOR_STATE_PATH)
    return state

print('✅', TARGET_NAME, 'controller loaded')


---
## ⚙️ 设计参数

本分支专用于孤立 helix32 与 40–60 aa minibinder。全部预算和采样数值在原参数区直接填写；默认 LARGE256，refine 已开启。无需选择折叠态上下文或核心热点。

默认使用 notebook 内置的已核验输入；可用 `NESPRIN_AF3_ZIP`、`NESPRIN_CANONICAL_FASTA` 指定；`NESPRIN_RUN_ROOT` 指定独立新运行目录。


In [ ]:
# 【可修改设置】输入模式设置：本分支使用已核验文件；未使用的兼容占位参数已注明。
# 所有可调参数 — 保留原参数区位置
# 设计任务类型；本分支为 binder。改变类型需要另一套输入与流程，不是预算调整。
design_task = 'binder'
# 原非 binder 模式的长度占位；本分支不读取，不决定 binder 长度。
len_min, len_max = 100, 200  # 原非 binder 模式占位，不参与本分支
# 原联网下载模式占位；当前 target_pdb=file，不使用此 PDB ID。
pdb_source = '1CRN'
# 原下载模式格式占位；当前使用包内核验输入，不执行下载。
pdb_download_format = 'cif'
# 原 motif 模式的链及起止编号占位；当前孤立 helix binder 分支不读取。
motif_chain, motif_start, motif_end = 'A', 1, 1
# 保留原 motif 模式兼容设置；当前实际设计规格在固定流程中构建。
fix_mode, mode = 'BKBN', 'unindex'
# 靶标输入来源；file 指向已核验片段，保持不变可保证生成/验证使用同一靶标。
target_pdb = 'file'
# 派生的已核验靶标链；不是自动选择其他结合表面的开关。
target_chain = TARGET_CHAIN
# 派生的片段文件位置，随运行目录自动准备；不需手工复制输入。
pdb_file_path = str(TARGET_FRAGMENT_PATH)
# None 表示不设置固定热点，整段 helix 可参与结合。
hotspot_residues = None
# 原对称设计模式占位；本 binder 分支不读取。
symmetry_type = 'C3'  # 原非 binder 模式占位
print('目标:', TARGET_NAME, '| 输入:', pdb_file_path)


In [ ]:
# 【可修改设置】研究参数：预算、seed、报告参考值、三轮 refine 与预览编号。
# ═══════════ 3. 通用参数（数值统一在这里修改）═══════════
# 固定靶标的 canonical 起止编号（两端均含）；换靶标需新分支，不能只改这两个数字。
TARGET_FRAGMENT_RANGE = (6646, 6677)
# 固定靶标的完整氨基酸序列，用于输入身份核验；不要为消除报错而改动。
EXPECTED_FRAGMENT_SEQ = 'MQCQDFHQLSQDLLLWLATAESRRQKAHVTSP'
# binder 长度边界，单位 aa；修改时须与下一行 left_min/left_max 同步。
BINDER_MIN_AA, BINDER_MAX_AA = 40, 60
# 实际生成 binder 的最短/最长长度，单位 aa；须与上行 BINDER_MIN_AA/BINDER_MAX_AA 一致。
left_min, left_max = 40, 60
# 第二段 binder 长度，单位 aa；当前为 0/0，保持单连续 binder。
right_min, right_max = 0, 0
# 额外 motif 长度，单位 aa；当前 0，不插入额外 motif。
motif_len = 0
# 从实际 binder 长度派生；无需手工修改。
length_min, length_max = left_min, left_max

# 主阶段每批请求的 RFD3 骨架数；8×32=256。改批量会改变显存需求。
diffusion_batch_size = 8
# 主阶段 RFD3 批次数；总预算=每批骨架数×批次数，不自动缩减。
n_batches = 32
# 主阶段扩散步数；200。改变会影响采样，不是只改变显示。
num_timesteps = 200
# RFD3 低显存开关；保留远程已完成版本 False。改为 True 可能改变资源与耗时。
low_memory_mode = False
# RFD3 原生采样步长缩放系数，沿用当前 2.5。
step_scale = 2.5
# RFD3 原生采样参数，沿用当前 0.35。
gamma_0 = 0.35
# RFD3 原生构象选项，沿用当前 False。
is_non_loopy = False
# 主随机种子；后续稳定阶段 seed 由它派生。恢复时必须保持一致。
random_seed = 20260922
seed_everything(random_seed)
# 0 处理全部请求骨架；正整数仅处理原编号前 N 个，不压缩无效骨架编号。
max_designs = 0  # 0=全部；不自动削减预算

# MPNN 引擎模式；当前 protein_mpnn，使用已有对应权重。
mpnn_model = 'protein_mpnn'
# 主阶段每个有效骨架的 MPNN 序列条数；默认 12。
mpnn_seqs_per_backbone = 12
# 主阶段每骨架验证序列数；0 验证全部 MPNN 序列，正整数限制前 N 条。
rf3_validations = 0  # 0=所有序列
# 每条序列请求的 RF3 模型数；默认 5，报告与导出以实际返回数为准。
rf3_models_per_sequence = 5
# 当前蛋白 binder 流程无配体，None；特殊体系不由本模板自动准备。
ligand = None
# 原生选择参数；当前 None，不引入固定热点。
select_hotspots = None
# 原生选择参数；当前 None，不添加埋藏表面限制。
select_buried = None
# 原生选择参数；当前 None，不添加暴露表面限制。
select_exposed = None
# 主阶段部分扩散参数；当前 None，主阶段完整扩散。refine 的部分扩散值见下方。
partial_t = None

# 固定分支标签，用于证据身份；不是输出路径或筛选开关。
SCHEME_TAG = 'SR56_A_helix32_minibinder_LARGE256_refine'
# 固定发布/缓存版本；不要手改来复用旧检查点，修改科研参数应创建新运行。
CACHE_SCHEMA = 'sr56-helix32-large256-refine-v3-20261002-chain-terminal-oxt-v1-20261004-output-v1-20261007-compact-v2-20261007'
# 固定数据有效性版本；不代表结合成功或新增生物学门槛。
VALIDATION_VERSION = 'sr56-isolated-output-validity-v3-20261002-chain-terminal-oxt-v1-20261004'

# 距离仅定义报告指标，不参与淘汰。
# 接触距离定义，单位 Å；只计算报告指标，不据此淘汰。
report_contact_distance_angstrom = 4.5
# 几何碰撞距离定义，单位 Å；只报告，区别于 RF3 原生碰撞标记。
report_clash_distance_angstrom = 2.0
# target fragment RMSD 图表参考刻度，单位 Å；不参与筛选。
report_fragment_rmsd_labels = (0.5, 1.0, 2.0)
# binder pose RMSD 图表参考刻度，单位 Å；不参与筛选。
report_pose_rmsd_labels = (1.0, 2.0, 3.0)
# pLDDT 报告参考线，内部标度 0–1；查看器显示 0–100，不参与筛选。
report_plddt_reference = 0.7
# 较高 pLDDT 报告参考线，内部标度 0–1；不参与筛选。
report_plddt_high_reference = 0.8
# pTM 报告参考线，标度 0–1；不参与筛选。
report_ptm_reference = 0.5
# PAE 报告参考线，单位 Å；不参与筛选。
report_pae_reference = 5.0
# ipTM 报告参考线，标度 0–1；不参与筛选。
report_iptm_high_reference = 0.8
# MPNN 序列恢复比例参考线，标度 0–1；不参与筛选。
report_mpnn_recovery_reference = 0.52
# RMSD 图表纵轴最小跨度，单位 Å；仅影响显示。
plot_rmsd_scale_min = 1.5

# ── Refinement 正式运行参数 ──
# 主阶段来源设计的 refine 起点数量；0 关闭，默认前 10，不足取实际数量。
refine_top_n = 10
# refine 最大轮数；默认 3。baseline 始终参与全循环最佳选择。
refine_rounds = 3
# 首轮部分扩散原生 t 参数，默认 1.8；不是模拟时间或 ns。需通过已安装 sampler 兼容检查。
refine_partial_t_start = 1.8
# 末轮部分扩散原生 t 参数，默认 0.5；中间轮按既有规则插值。
refine_partial_t_end = 0.5
# 每轮 refine 扩散步数，默认 150。
refine_timesteps = 150
# 每个 refine 有效骨架的 MPNN 序列数，默认 8。
refine_mpnn_seqs = 8
# 每个 refine 骨架验证序列数；0 验证全部，正整数限制前 N 条。
refine_rf3_validations = 0
# 每个 refine 父路径每批请求的骨架数，默认 1。
refine_backbone_batch = 1
# 每个 refine 父路径的批次数，默认 8；骨架上限=本项×上项。
refine_batch = 8
# 每轮最多保留的不同来源骨架路径数，默认 3；不添加比例门槛。
refine_beam_k = 3
# 不保留 beam 比例门槛、RMSD 收敛门槛或未实现的重采样开关。
# 仅控制预览：批次和批内骨架均从 0 编号；不改变计算与正式排名。
preview_batch, preview_in_batch = 0, 0
shortpep_monitor_update('CONFIG_READY', target=str(TARGET_FRAGMENT_PATH),
                        canonical_range=list(TARGET_FRAGMENT_RANGE), input_zip=str(AF3_SR54_56_ZIP))
print(f'骨架 {diffusion_batch_size*n_batches} | 每骨架序列 {mpnn_seqs_per_backbone} | refine {refine_top_n}×{refine_rounds}')


In [ ]:
# 【固定流程 · 无需修改】检查输入、长度、预算与接口约束；不添加生物学筛选门槛。
# ── 参数校验：仅输入、预算与接口约束 ──
assert design_task == 'binder'
assert target_pdb == 'file' and Path(pdb_file_path) == TARGET_FRAGMENT_PATH
assert hotspot_residues is None and select_hotspots is None and TARGET_HOTSPOTS is None
assert len(EXPECTED_FRAGMENT_SEQ) == TARGET_FRAGMENT_RANGE[1]-TARGET_FRAGMENT_RANGE[0]+1 == 32
assert (length_min, length_max) == (BINDER_MIN_AA, BINDER_MAX_AA) == (40, 60)
assert right_min == right_max == 0
for _p in (diffusion_batch_size, n_batches, num_timesteps, mpnn_seqs_per_backbone,
           rf3_models_per_sequence, refine_rounds, refine_timesteps,
           refine_mpnn_seqs, refine_backbone_batch, refine_batch, refine_beam_k):
    assert isinstance(_p, int) and _p > 0
for _p in (max_designs, rf3_validations, refine_top_n, refine_rf3_validations):
    assert isinstance(_p, int) and _p >= 0
assert 0 < report_clash_distance_angstrom < report_contact_distance_angstrom
assert 0 < refine_partial_t_end <= refine_partial_t_start <= 15
assert np.isfinite(step_scale) and np.isfinite(gamma_0)
total_designs = diffusion_batch_size * n_batches
n_proc = min(max_designs, total_designs) if max_designs else total_designs
contig_str = None
print('✅ 参数校验通过；接触、碰撞、置信度和 RMSD 只报告')
print(f'RFD3 {diffusion_batch_size}×{n_batches}={total_designs}；binder {length_min}–{length_max} aa；seed {random_seed}')


---
### 输入预检与来源

核对 Q6ZWQ0-1 canonical 序列、AF3 model_3 散列、6646–6677 映射和完整主链；提取同一 32 aa 片段。二级结构和接触仅作报告，不设生物学门槛。


In [ ]:
# 【固定流程 · 无需修改】提取靶标片段并核对序列、canonical 编号、链及主链原子。
# SR56-A isolated 32 aa input: source identity, canonical mapping and shared checks.
from pathlib import Path
from zipfile import ZipFile
from biotite.structure import annotate_sse, concatenate
from scipy.spatial import cKDTree
import hashlib, gc

def _json_value(value):
    if isinstance(value, dict): return {str(k): _json_value(v) for k,v in value.items()}
    if isinstance(value, (tuple, list)): return [_json_value(v) for v in value]
    if isinstance(value, np.ndarray): return _json_value(value.tolist())
    if isinstance(value, np.generic): return _json_value(value.item())
    if isinstance(value, float) and not np.isfinite(value): return None
    if isinstance(value, Path): return str(value)
    return value

def _write_json_atomic(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(_json_value(payload), ensure_ascii=False, indent=1, allow_nan=False))
    os.replace(temp, path)

def _check_checkpoint_identity(rec):
    if not isinstance(rec, dict) or rec.get('schema') != CACHE_SCHEMA or rec.get('run_namespace') != _RUN_NS:
        raise RuntimeError('旧 schema / 其他运行命名空间的断点，拒绝复用')

def _stage_record(path, key, fingerprint):
    rec = _resume.load_stage(path, key, missing='restart')
    if rec is not None and rec.get('cfg') != fingerprint:
        raise RuntimeError(f'阶段 {key} 配置/输入身份改变；请使用新的运行目录')
    return rec

def _stage_seed(*parts):
    text = json.dumps([random_seed, *parts], ensure_ascii=False, separators=(',', ':'))
    return int(hashlib.sha256(text.encode()).hexdigest()[:8], 16) % (2**31)

def _seed_engine(engine, seed):
    # Weight/pipeline initialization may consume RNG: finish it before setting a stage seed.
    initialize = getattr(engine, 'initialize', None)
    if callable(initialize): initialize()
    seed_everything(seed)
    engine.seed = seed

def _release_engines():
    # Release all notebook aliases; empty_cache alone does not release live weights.
    for name in ('model', 'mpnn', 'rf3_engine', 'rfd3_model', 'mpnn_eng', 'rf3_eng'):
        globals()[name] = None
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def _require_cuda():
    if not torch.cuda.is_available():
        raise RuntimeError('本运行环境 CUDA 不可用；文件/CPU 核验不能替代 GPU 推理')

def _number(value):
    return float(value) if isinstance(value, (int,float,np.number)) and not isinstance(value, (bool,np.bool_)) else float('nan')

def _report_summary(summary):
    out = dict(summary or {})
    for key in ('ranking_score','overall_plddt','ptm','iptm','overall_pae'):
        out[key] = _number(out.get(key))
    return out

def _rank_key(summary, stable_id=()):
    value = _number((summary or {}).get('ranking_score'))
    return (int(np.isfinite(value)), value if np.isfinite(value) else 0.0,
            tuple(-int(i) for i in stable_id))

def _metric_csv(value):
    if value is None or (isinstance(value, (float,np.floating)) and not np.isfinite(value)): return ''
    if isinstance(value, (dict,list,tuple)): return json.dumps(_json_value(value), ensure_ascii=False)
    return value

QC_METRIC_FIELDS = ('binder_self_rmsd_A', 'binder_pose_rmsd_A', 'target_fragment_rmsd_A',
                    'target_alpha_fraction', 'target_contact_residues', 'contact_atom_pairs',
                    'min_distance_A', 'has_geometric_clash', 'contact_residue_ids')

def _empty_metrics():
    return {k: float('nan') for k in QC_METRIC_FIELDS}

assert AF3_SR54_56_ZIP.exists() and CANONICAL_FASTA_PATH.exists(), '缺少核验输入来源'
_fasta_lines = CANONICAL_FASTA_PATH.read_text().splitlines()
assert _fasta_lines[0].startswith('>sp|Q6ZWQ0|') and 'SV=2' in _fasta_lines[0]
_canonical_seq = ''.join(_fasta_lines[1:]).replace(' ', '')
assert len(_canonical_seq) == 6874
assert hashlib.sha256(_canonical_seq.encode()).hexdigest() == '829e8b2444a7d3d4d6e430a71700fb717082970ddee10381d8f2202e828fdf96'
assert _canonical_seq[TARGET_FRAGMENT_RANGE[0]-1:TARGET_FRAGMENT_RANGE[1]] == EXPECTED_FRAGMENT_SEQ
with ZipFile(AF3_SR54_56_ZIP) as _zf:
    _request = json.loads(_zf.read('fold_nesprin_sr54_sr56_job_request.json'))
    _request_seq = _request[0]['sequences'][0]['proteinChain']['sequence']
    _model_member = 'fold_nesprin_sr54_sr56_model_3.cif'
    _cif_bytes = _zf.read(_model_member)
assert hashlib.sha256(_cif_bytes).hexdigest() == '7dd813522be36fdb40daf16b8684c30e6711039f02497caab374999636eea89d'
# Parent construct is used only to establish the fragment's canonical mapping.
assert len(_request_seq) == 384
assert _request_seq[:348] == _canonical_seq[6432:6780]
_model_cif = PREP_DIR / _model_member
_model_cif.write_bytes(_cif_bytes)
_raw_aa = _get_structure(_CIFFile.read(str(_model_cif)), model=1, use_author_fields=False)
assert len(_chain_list(_raw_aa)) == 1 and _chain_seq(_raw_aa, _chain_list(_raw_aa)[0][0]) == _request_seq
_raw_st = get_residue_starts(_raw_aa, add_exclusive_stop=True)
assert len(_raw_st)-1 == 384
_lo, _hi = TARGET_FRAGMENT_RANGE
_start, _stop = _lo-6433, _hi-6433+1
TARGET_FRAGMENT_AA = _raw_aa[_raw_st[_start]:_raw_st[_stop]].copy()
TARGET_FRAGMENT_AA.chain_id[:] = TARGET_CHAIN
_target_st = get_residue_starts(TARGET_FRAGMENT_AA, add_exclusive_stop=True)
for _i in range(32): TARGET_FRAGMENT_AA.res_id[_target_st[_i]:_target_st[_i+1]] = _lo+_i
TARGET_FRAGMENT_SEQ = _chain_seq(TARGET_FRAGMENT_AA, TARGET_CHAIN)
assert TARGET_FRAGMENT_SEQ == EXPECTED_FRAGMENT_SEQ
assert _backbone_coord_list(TARGET_FRAGMENT_AA, TARGET_CHAIN).shape == (128,3)
assert np.isfinite(TARGET_FRAGMENT_AA.coord).all()
TARGET_SEQ_SHA256 = hashlib.sha256(TARGET_FRAGMENT_SEQ.encode()).hexdigest()
to_cif_file(TARGET_FRAGMENT_AA, str(TARGET_FRAGMENT_PATH))
_provenance = {'uniprot':TARGET_UNIPROT, 'species':TARGET_SPECIES, 'chain':TARGET_CHAIN,
               'canonical_range':list(TARGET_FRAGMENT_RANGE), 'sequence':TARGET_FRAGMENT_SEQ,
               'sequence_sha256':TARGET_SEQ_SHA256, 'source_model_member':_model_member,
               'model_cif_sha256':hashlib.sha256(_cif_bytes).hexdigest(),
               'source_zip_sha256':_sha256_file(AF3_SR54_56_ZIP),
               'canonical_full_sequence_sha256':hashlib.sha256(_canonical_seq.encode()).hexdigest(),
               'target_cif_sha256':_sha256_file(TARGET_FRAGMENT_PATH),
               'coordinate_source':'unmodified AF3 model_3 residues, native 32 aa construct',
               'canonical_to_construct_positions':list(range(_start+1,_stop+1))}
_write_json_atomic(TARGET_FRAGMENT_PATH.with_suffix('.provenance.json'), _provenance)

def _resolve_target_chain(aa):
    hits = [c for c,_ in _chain_list(aa) if _chain_seq(aa,c) == TARGET_FRAGMENT_SEQ]
    return hits[0] if len(hits)==1 else None

def _target_kind(aa):
    return 'fragment32' if _resolve_target_chain(aa) is not None else None

def _single_binder(aa):
    tc = _resolve_target_chain(aa)
    chains = [c for c,_ in _chain_list(aa)]
    if tc is None or len(chains)!=2 or len(set(chains))!=2: raise ValueError('target_binder_chain_mapping')
    return aa[_chain_strs(aa.chain_id) == next(c for c in chains if c!=tc)]

def _heavy_mask(aa):
    el = np.char.upper(np.asarray(aa.element).astype(str))
    an = np.char.upper(np.asarray(aa.atom_name).astype(str))
    placeholder = (an=='OXT') & np.isnan(aa.coord).all(axis=1)
    return (el!='H') & (el!='D') & (~np.char.startswith(an,'H')) & (~placeholder)

def _validate_complex(aa, expected_binder_seq=None):
    errors = []; metrics = _empty_metrics()
    try:
        chains = [c for c,_ in _chain_list(aa)]
        tc = _resolve_target_chain(aa)
        if len(chains)!=2 or len(set(chains))!=2 or tc is None:
            errors.append('target_binder_identity')
        else:
            binder = _single_binder(aa)
            seq = _binder_seq_of(aa)
            if not BINDER_MIN_AA <= len(seq) <= BINDER_MAX_AA: errors.append('binder_length')
            if expected_binder_seq is not None and seq != expected_binder_seq: errors.append('binder_seq_mismatch')
            if _backbone_coord_list(aa) is None: errors.append('backbone_missing_duplicate_or_nonfinite')
            st = get_residue_starts(aa, add_exclusive_stop=True)
            for a,b in zip(st[:-1], st[1:]):
                if len(set(aa.atom_name[a:b])) != b-a: errors.append('duplicate_atom_identity'); break
            for chain in chains:
                ids = aa.res_id[get_residue_starts(aa)][_chain_strs(aa.chain_id[get_residue_starts(aa)])==chain]
                if len(set(map(int,ids)))!=len(ids): errors.append('duplicate_residue_identity')
            # Exempt all-NaN OXT only in each chain's final residue block, including target.
            allowed = np.zeros(len(aa), dtype=bool)
            for chain in chains:
                a,b = next((a,b) for a,b in reversed(list(zip(st[:-1],st[1:]))) if _chain_strs(aa.chain_id)[a]==chain)
                allowed[a:b] = (aa.atom_name[a:b]=='OXT') & np.isnan(aa.coord[a:b]).all(axis=1)
            if not np.isfinite(aa.coord[~allowed]).all(): errors.append('nonfinite_actual_atom_coords')
    except Exception as exc:
        errors.append('structure_data_error:'+type(exc).__name__)
    return {'valid_output':not errors, 'errors':list(dict.fromkeys(errors)), 'metrics':metrics}

def _interface_stats(aa, binder=None):
    tc = _resolve_target_chain(aa)
    if tc is None: raise ValueError('fragment_identity')
    target = aa[_chain_strs(aa.chain_id)==tc]
    binder = _single_binder(aa) if binder is None else binder
    tx = np.asarray(target.coord[_heavy_mask(target)],float)
    bx = np.asarray(binder.coord[_heavy_mask(binder)],float)
    if not len(tx) or not len(bx) or not np.isfinite(tx).all() or not np.isfinite(bx).all(): raise ValueError('nonfinite_contact_coords')
    st = get_residue_starts(target, add_exclusive_stop=True)
    atom_ids = np.repeat(np.arange(32)+TARGET_FRAGMENT_RANGE[0], np.diff(st))[_heavy_mask(target)]
    tree = cKDTree(tx)
    near = tree.query_ball_point(bx, report_contact_distance_angstrom)
    touched = sorted({int(atom_ids[i]) for hits in near for i in hits})
    dmin = float(np.min(tree.query(bx,k=1)[0]))
    return {'target_contact_residues':len(touched), 'contact_atom_pairs':sum(map(len,near)),
            'min_distance_A':dmin, 'has_geometric_clash':dmin<report_clash_distance_angstrom,
            'contact_residue_ids':touched}

def _fragment_rmsd(aa):
    return _aligned_rmsd(_backbone_coord_list(TARGET_FRAGMENT_AA,TARGET_CHAIN),
                         _backbone_coord_list(aa,_resolve_target_chain(aa)))

def _target_alpha_fraction(aa):
    tc = _resolve_target_chain(aa)
    target = aa[_chain_strs(aa.chain_id)==tc]
    return float(np.mean(np.asarray(annotate_sse(target))=='a'))

def _assess_model(aa_ref, model_aa, summary, expected_binder_seq=None):
    result = _validate_complex(model_aa, expected_binder_seq)
    if not result['valid_output']: return result
    metrics = result['metrics']
    operations = {'binder_self_rmsd_A':lambda:_binder_self_rmsd(aa_ref,model_aa),
                  'binder_pose_rmsd_A':lambda:_binder_backbone_rmsd(aa_ref,model_aa),
                  'target_fragment_rmsd_A':lambda:_fragment_rmsd(model_aa),
                  'target_alpha_fraction':lambda:_target_alpha_fraction(model_aa)}
    for key, operation in operations.items():
        try: metrics[key] = operation()
        except (ValueError,IndexError,TypeError): metrics[key] = float('nan')
    metrics.update(_interface_stats(model_aa))
    # No score, contact, clash, alpha fraction or RMSD threshold changes valid_output.
    return result

def _complex_components(aa, eid):
    check = _validate_complex(aa)
    if not check['valid_output']: raise RuntimeError('RF3 input data invalid: '+','.join(check['errors']))
    tc = _resolve_target_chain(aa); bc = str(_single_binder(aa).chain_id[0])
    return {'name':str(eid), 'components':[
        {'chain_id':tc,'seq':TARGET_FRAGMENT_SEQ,'chain_type':'polypeptide(L)'},
        {'chain_id':bc,'seq':_chain_seq(aa,bc),'chain_type':'polypeptide(L)'}]}

def _hotspot_rmsd_map(aa_ref, aa_mov, input_aa=None, hotspot_residues=None):
    # Original display interface retained; the value covers all 32 aa, not a hotspot.
    return {'fragment32':_fragment_rmsd(aa_mov)}

def _anchor_contacts_from_complex(aa, input_aa=None, hotspot_residues=None, cutoff=None):
    return {'fragment32':_interface_stats(aa)}

def _reset_fragment_target(aa):
    check = _validate_complex(aa)
    if not check['valid_output']: raise RuntimeError('refine input invalid: '+','.join(check['errors']))
    rt = _target_frame_transform(TARGET_FRAGMENT_AA,aa)
    if rt is None: raise RuntimeError('refine: fragment backbone frame cannot be determined')
    binder = _single_binder(_transform_coords(aa,rt)).copy()
    binder.chain_id[:] = 'B' if TARGET_CHAIN!='B' else 'C'
    return concatenate((TARGET_FRAGMENT_AA.copy(),binder))

def _check_mpnn_backbone(backbone, result):
    if _resolve_target_chain(result) is None: raise RuntimeError('MPNN changed fixed target identity')
    before, after = _backbone_coord_list(backbone), _backbone_coord_list(result)
    if before is None or after is None or before.shape!=after.shape or not np.allclose(before,after,atol=1e-3,rtol=0):
        raise RuntimeError('MPNN backbone differs from generated backbone')

def _current_cfg():
    keys = ('design_task','target_chain','left_min','left_max','right_min','right_max',
            'diffusion_batch_size','n_batches','num_timesteps','low_memory_mode','step_scale',
            'gamma_0','is_non_loopy','random_seed','max_designs','mpnn_model',
            'mpnn_seqs_per_backbone','rf3_validations','rf3_models_per_sequence',
            'report_contact_distance_angstrom','report_clash_distance_angstrom','SCHEME_TAG')
    weights = {}
    for key in ('RFD3_CKPT','MPNN_CKPT','RF3_CKPT'):
        path = globals().get(key)
        weights[key] = {'path':str(path), 'identity':_weight_identity(path) if path and Path(path).exists() else None}
    return {'schema':CACHE_SCHEMA, 'validation_version':VALIDATION_VERSION, 'python':sys.version.split()[0],
            'parameters':{k:globals().get(k) for k in keys}, 'engine_sources':_engine_sources(),
            'weights':weights, 'target_range':list(TARGET_FRAGMENT_RANGE),
            'target_sequence_sha256':TARGET_SEQ_SHA256, 'target_cif_sha256':_sha256_file(TARGET_FRAGMENT_PATH),
            'source_model_sha256':_provenance['model_cif_sha256'], 'seed_rule':'sha256(primary_seed,stage_ids)-v1'}

def _module_fp(module):
    cfg = _current_cfg()
    if module=='rfd3':
        keys = ('diffusion_batch_size','n_batches','num_timesteps','low_memory_mode','step_scale','gamma_0',
                'is_non_loopy','random_seed','left_min','left_max')
        record = {'target':cfg['target_cif_sha256'], 'sequence':TARGET_SEQ_SHA256,
                  'parameters':{k:cfg['parameters'][k] for k in keys},
                  'engine':cfg['engine_sources'], 'weight':cfg['weights']['RFD3_CKPT']}
    elif module=='mpnn':
        record = {'rfd3':_module_fp('rfd3'), 'valid_design_indices':globals().get('_valid_design_indices',[]),
                  'max_designs':max_designs,'model':mpnn_model,'sequences':mpnn_seqs_per_backbone,
                  'backbones':[(p.name,_sha256_file(p)) for p in sorted(RFD3_DIR.glob('rfd3_*.cif'))],
                  'engine':cfg['engine_sources'],'weight':cfg['weights']['MPNN_CKPT']}
    elif module=='rf3':
        record = {'mpnn':_module_fp('mpnn'), 'models':rf3_models_per_sequence, 'validation_version':VALIDATION_VERSION,
                  'sequence':TARGET_SEQ_SHA256,'engine':cfg['engine_sources'],'weight':cfg['weights']['RF3_CKPT']}
    else: raise ValueError(module)
    return _config_fingerprint({'cache_schema':CACHE_SCHEMA, 'seed_rule':cfg['seed_rule'],'stage':module,'record':record})

_fix.hotspot_rmsd = _hotspot_rmsd_map
_fix.anchor_contacts_from_complex = _anchor_contacts_from_complex
_preflight_report = {'sequence_length':32, 'canonical_range':list(TARGET_FRAGMENT_RANGE),
                     'sequence_sha256':TARGET_SEQ_SHA256, 'model_sha256':_provenance['model_cif_sha256'],
                     'target_alpha_fraction':_target_alpha_fraction(TARGET_FRAGMENT_AA),
                     'interpretation':'isolated fragment; descriptive alpha fraction, no folded-state test'}
_write_json_atomic(PREP_DIR/'fragment_preflight.json',_preflight_report)
shortpep_monitor_update('PREFLIGHT_PASS',target=str(TARGET_FRAGMENT_PATH),**_preflight_report)
print('✅ 孤立 helix32 来源/映射预检通过:',_preflight_report)


In [ ]:
# 【固定流程 · 无需修改】准备六个阶段目录、报告目录与运行状态目录。
# ── 资源估算与独立输出目录 ──
main_pair_limit = n_proc * (mpnn_seqs_per_backbone if rf3_validations==0 else min(mpnn_seqs_per_backbone,rf3_validations))
_refine_paths = 1 + max(0,refine_rounds-1)*refine_beam_k
_refine_sequences = refine_mpnn_seqs if refine_rf3_validations==0 else min(refine_mpnn_seqs,refine_rf3_validations)
refine_pair_limit = min(refine_top_n,n_proc)*_refine_paths*refine_backbone_batch*refine_batch*_refine_sequences
print(f'📊 RFD3 {total_designs}；MPNN 最多 {n_proc*mpnn_seqs_per_backbone}；RF3 主 {main_pair_limit} + refine {refine_pair_limit} = {main_pair_limit+refine_pair_limit} 条')
print(f'每条配置 {rf3_models_per_sequence} 模型，实际返回数单独记账；GPU 耗时/峰值显存未实测')
_RUN_NS = 'sr56_a_helix32_minibinder_large256_refine_v3_20261002_chain_terminal_oxt_v1_20261004_output_v1_20261007_compact_v2_20261007_layout_v3'
pdb_tag = Path(pdb_file_path).stem
OUT_DIR = PROJECT_ROOT
MAIN_DIR = OUT_DIR/'main'
RFD3_DIR, MPNN_DIR, RF3_DIR = (MAIN_DIR/name for name in ('rfd3','mpnn','rf3'))
REFINE_DIR = REPORT_DIR/'refine'
REFINE_RFD3_DIR, REFINE_MPNN_DIR, REFINE_RF3_DIR = (OUT_DIR/'refine'/name for name in ('rfd3','mpnn','rf3'))
for _d in (OUT_DIR,RFD3_DIR,MPNN_DIR,RF3_DIR,REFINE_DIR,REFINE_RFD3_DIR,REFINE_MPNN_DIR,REFINE_RF3_DIR): _d.mkdir(parents=True,exist_ok=True)
CKPT_FILE = CHECKPOINT_DIR/'main/checkpoint.json'
print('输出目录:',OUT_DIR)
# SR56_FEEDBACK_BEGIN
_feedback.configure(config={name:globals().get(name) for name in ['n_batches', 'diffusion_batch_size', 'n_proc', 'mpnn_seqs_per_backbone', 'rf3_validations', 'rf3_models_per_sequence', 'refine_top_n', 'refine_rounds', 'refine_beam_k', 'refine_backbone_batch', 'refine_batch', 'refine_mpnn_seqs', 'refine_rf3_validations', 'random_seed', 'main_pair_limit', 'refine_pair_limit', 'total_designs', 'low_memory_mode', 'num_timesteps', 'step_scale', 'gamma_0', 'refine_timesteps', 'refine_partial_t_start', 'refine_partial_t_end', 'design_task', 'target_chain', 'left_min', 'left_max', 'right_min', 'right_max', 'is_non_loopy', 'max_designs', 'mpnn_model', 'report_contact_distance_angstrom', 'report_clash_distance_angstrom', 'SCHEME_TAG', 'CACHE_SCHEMA', 'VALIDATION_VERSION']}, target={'canonical_range':TARGET_FRAGMENT_RANGE, 'sequence':TARGET_FRAGMENT_SEQ, 'binder_range':[BINDER_MIN_AA,BINDER_MAX_AA], 'chain':'A', 'sequence_sha256':globals().get('TARGET_SEQ_SHA256'), 'source_model_sha256':globals().get('_provenance',{}).get('model_cif_sha256')}, namespace=_RUN_NS, out_dir=OUT_DIR)
# SR56_FEEDBACK_END


---
## 第 1 步：RFD3 模体条件骨架生成

RFdiffusion3 (RFD3) 通过扩散模型，在固定模体的条件下生成 *de novo* 蛋白质骨架。

**本步流程：**
1. 自动下载 / 定位 PDB 结构
2. 📷 预览输入模体
3. 构建设计规格（contig / unindex）
4. 运行 RFD3 扩散生成
5. 收集所有设计，写入 CIF；在预览 Cell 中输入批次号和批内序号预览指定设计

**输出：** 每个设计一个 `RFD3Output`，包含 `atom_array`（Biotite AtomArray）等。

In [ ]:
# 【固定流程 · 无需修改】执行主阶段 RFD3，逐批提交骨架与断点。
# —— 第 1 步：RFD3 — 模体条件骨架生成 ——
# ── 1a. 获取 PDB ──
print("━" * 50)
print("第 1 步：RFD3 — 模体条件骨架生成")
print("━" * 50)
t0 = time.time()
def resolve_pdb(source, file_path, fmt="pdb"):
    """返回结构文件绝对路径，必要时从 RCSB 下载。
    fmt: "pdb" / "cif"（仅 source != "file" 时生效）
    本地文件支持 .pdb / .cif / .mmcif 后缀（RFD3 内部按内容解析）。"""
    if source != "file":
        ext = "cif" if fmt == "cif" else "pdb"
        dest = Path(f"{source}.{ext}").resolve()
        if not dest.exists():
            print(f"  下载 {source}.{ext} ...")
            try:
                urllib.request.urlretrieve(
                    f"https://files.rcsb.org/download/{source}.{ext}", str(dest)
                )
            except Exception as e:
                raise RuntimeError(
                    f"下载 {source}.{ext} 失败: {e}\n"
                    f"请检查网络连接，或手动下载后放到: {dest}"
                ) from e
            if dest.stat().st_size < 1000:
                dest.unlink(missing_ok=True)
                raise RuntimeError(
                    f"下载 {source}.{ext} 内容异常（可能 PDB ID 不存在），已删除残留文件"
                )
        else:
            print(f"  ✓ {dest} 已存在")
        return str(dest)
    else:
        p = Path(file_path).resolve()
        if not p.exists():
            raise FileNotFoundError(
                f"结构文件未找到: {file_path}（支持 .pdb / .cif / .mmcif）"
            )
        return str(p)

pdb_path = resolve_pdb(
    target_pdb if design_task == "binder" else pdb_source,
    pdb_file_path, pdb_download_format,
)
print(f"结构文件: {pdb_path}（格式: {(Path(pdb_path).suffix[1:] or 'unknown').upper()}）")

# ── 📷 预览输入结构 ──
from rfd3.utils.inference import inference_load_
_input_dict = inference_load_(pdb_path)   # 使用外部 CCD 镜像（CCD_MIRROR_PATH）
if "atom_array" not in _input_dict:
    raise RuntimeError("inference_load_ 返回格式异常，缺少 'atom_array' 键（RFD3 版本可能不兼容）")
input_aa = _input_dict["atom_array"]
print(f"\n 输入模体: {len(input_aa)} 原子, {len(set(input_aa.res_id))} 残基")
print("📷 输入结构 3D 预览")
display(view(input_aa))

# ── binder：选定靶标链身份/连续编号校验（cell11 以 A<min>-<max> 建 contig）──
if design_task == "binder":
    _st10 = get_residue_starts(input_aa)
    _ch10 = _chain_strs(input_aa.chain_id[_st10])
    _sel10 = _ch10 == target_chain
    _ids10 = sorted(int(r) for r in input_aa.res_id[_st10][_sel10])
    if not _ids10:
        raise ValueError("靶标链 " + str(target_chain) + " 不存在于输入结构（链: " + str(sorted(set(_ch10))) + "）")
    if _ids10 != list(range(_ids10[0], _ids10[0] + len(_ids10))):
        _gaps10 = [(_ids10[i], _ids10[i + 1]) for i in range(len(_ids10) - 1) if _ids10[i + 1] != _ids10[i] + 1]
        raise ValueError("靶标链 " + str(target_chain) + " 编号不连续（缺口 " + str(_gaps10[:5]) + "）；cell11 contig 假设连续编号，拒绝静默生成错误 contig")
    print("  ✓ 靶标链 " + str(target_chain) + ": " + str(len(_ids10)) + " 残基，编号 " + str(_ids10[0]) + "-" + str(_ids10[-1]) + " 连续")

# ── 模体残基存在性校验（提前暴露参数错误）──
if design_task in ("motif_scaffolding", "ligand_design"):
    _chain_res = set(input_aa.res_id[input_aa.chain_id == motif_chain])
    _missing = [r for r in range(motif_start, motif_end + 1) if r not in _chain_res]
    if _missing:
        _avail = sorted(_chain_res)
        raise ValueError(
            f"模体残基 {motif_chain}{_missing[0]} 不存在于输入结构！\n"
            f"链 {motif_chain} 残基范围: {_avail[0]}-{_avail[-1]}（共 {len(_avail)} 个）\n"
            f"请修正 motif_start/motif_end 或 motif_chain"
        )
    print(f"  ✓ 模体 {motif_chain}{motif_start}-{motif_end} 存在性校验通过")


In [ ]:
# 【固定流程 · 无需修改】核对生成骨架的链、身份及坐标有效性。
# ── 1b. 构建设计规格；同一完整 32 aa 靶标 + 单连续 binder ──
from rfd3.inference.input_parsing import DesignInputSpecification
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
assert _resolve_target_chain(input_aa) == target_chain
assert len(get_residue_starts(input_aa)) == 32
assert list(input_aa.res_id[get_residue_starts(input_aa)]) == list(range(*[TARGET_FRAGMENT_RANGE[0],TARGET_FRAGMENT_RANGE[1]+1]))
_tgt_contig = f'{target_chain}{TARGET_FRAGMENT_RANGE[0]}-{TARGET_FRAGMENT_RANGE[1]}'
spec_kwargs = {'input':str(pdb_file_path), 'contig':f'{_tgt_contig},/0,{left_min}-{left_max}',
               'select_fixed_atoms':{_tgt_contig:'ALL'}, 'is_non_loopy':is_non_loopy}
spec = DesignInputSpecification(**spec_kwargs)

# RFD3_API_COMPAT_BEGIN
# Foundry 0.1.9 accepts named JSON input, but its object route calls item.keys().
# Keep the validated original sparse fields; do not dump unrelated Pydantic defaults.
_spec_path = PREP_DIR / 'rfd3_main_input.json'
_write_json_atomic(_spec_path, {'design_spec':spec_kwargs})
_spec_path = str(_spec_path.resolve())

# Pure CPU, deterministic compatibility check; no weights, RNG or budget changes.
# Upstream PR #138 fixes partial_t filtering. A version label alone is insufficient.
def _rfd3_partial_preflight():
    report = {'schema':'sr56-rfd3-compatibility-v1','status':'NOT_REQUIRED',
              'refine_requested':bool(refine_top_n > 0 and refine_rounds > 0),
              'input_transport':'named_json_path','partial_checks':[]}
    if report['refine_requested']:
        try:
            import inspect as _compat_inspect
            from rfd3.model.inference_sampler import SampleDiffusionWithMotif
            sampler = SampleDiffusionWithMotif(num_timesteps=refine_timesteps,
                                               step_scale=step_scale,gamma_0=gamma_0)
            method = sampler._construct_inference_noise_schedule
            source = _compat_inspect.getsourcefile(method)
            report['scheduler_source'] = source
            report['scheduler_sha256'] = _sha256_file(source) if source else None
            full = np.asarray(method(device=torch.device('cpu')).tolist(), dtype=float)
            if full.ndim != 1 or len(full) < 2 or not np.isfinite(full).all() or (np.diff(full) > 0).any():
                raise ValueError('invalid full noise schedule')
            report['full_steps'] = len(full)
            for value in (refine_partial_t_start, refine_partial_t_end):
                value = float(value)
                actual = np.asarray(method(device=torch.device('cpu'),
                                partial_t=torch.tensor(value,device='cpu')).tolist(),dtype=float)
                expected = full[full <= value]
                if not len(expected): expected = full[-1:]
                if actual.shape != expected.shape or not np.isfinite(actual).all() or not np.allclose(actual,expected,rtol=1e-5,atol=1e-7):
                    raise ValueError(f'partial_t={value}: returned schedule differs from filtered full schedule')
                report['partial_checks'].append({'partial_t':value,'steps':len(actual),
                                      'max_noise':float(actual.max()),'min_noise':float(actual.min())})
            report['status'] = 'PASS'
        except Exception as exc:
            report.update(status='BLOCKED',reason_code='RFD3_PARTIAL_T_INCOMPATIBLE',error=str(exc))
            _write_json_atomic(PREP_DIR/'rfd3_compatibility.json',report)
            raise RuntimeError('RFD3_PARTIAL_T_INCOMPATIBLE: installed partial diffusion failed CPU preflight; '
                               'verify upstream PR #138 in a separate scientific environment. '
                               'No model was loaded; refine parameters were not changed.') from exc
    _write_json_atomic(PREP_DIR/'rfd3_compatibility.json',report)
    return report

_rfd3_compatibility = _rfd3_partial_preflight()
# RFD3_API_COMPAT_END

# ── 1c. 逐批保存，保留请求编号；未返回的骨架使用 None 占位 ──
_release_engines()
_rfd3_fp = _module_fp('rfd3')
all_designs = []

# SR56_DISPLAY_BEGIN
_sr56_tick('RFD3', False, 0, 0, reset=True)
# SR56_DISPLAY_END
try:
    for b in range(n_batches):
        key = f'rfd3_batch_{b}'
        seed = _stage_seed('main','rfd3',b)
        fp = _config_fingerprint({'cfg':_rfd3_fp,'batch':b,'seed':seed})
        # SR56_FEEDBACK_BEGIN
        _feedback_safe(lambda: _feedback.event('START', 'main_rfd3', key=key))
        # SR56_FEEDBACK_END
        rec = _stage_record(CKPT_FILE,key,fp)

        # SR56_DISPLAY_BEGIN
        _sr56_display_reused = rec is not None
        # SR56_DISPLAY_END
        # SR56_FEEDBACK_BEGIN
        _feedback_safe(lambda: _feedback.event('REUSED', 'main_rfd3', key=key, **rec)) if rec is not None else None
        # SR56_FEEDBACK_END
        if rec is None:
            _require_cuda()
            if model is None:
                _quiet_logging()
                model = RFD3InferenceEngine(**RFD3InferenceConfig(
                    ckpt_path=RFD3_CKPT, diffusion_batch_size=diffusion_batch_size,
                    low_memory_mode=low_memory_mode,
                    inference_sampler={'num_timesteps':num_timesteps,'step_scale':step_scale,'gamma_0':gamma_0}))
            _seed_engine(model,seed)
            outs = model.run(inputs=_spec_path,out_dir=None,n_batches=1)
            raw = [out for group in (outs or {}).values() for out in group]
            if len(raw)>diffusion_batch_size: raise RuntimeError('RFD3 返回数量超过本批预算')
            paths = []
            for i, out in enumerate(raw):
                p = RFD3_DIR/f'rfd3_{pdb_tag}_B{b}_D{i}.cif'
                to_cif_file(out.atom_array,str(p)); paths.append(str(p))
            rec = {'cfg':fp,'seed':seed,'paths':paths,'n_requested':diffusion_batch_size,'n_generated':len(paths)}
            _resume.save_stage(CKPT_FILE,key,rec,files=paths)
            # SR56_FEEDBACK_BEGIN
            _feedback_safe(lambda: _feedback.event('COMMITTED', 'main_rfd3', key=key, **rec))
            # SR56_FEEDBACK_END
        for i in range(diffusion_batch_size):
            aa = _read_cif(rec['paths'][i]) if i<len(rec['paths']) else None
            all_designs.append((f'{b}_d{i}',aa,len(get_residue_starts(aa)) if aa is not None else 0))

        # SR56_DISPLAY_BEGIN
        _sr56_tick('RFD3', _sr56_display_reused, diffusion_batch_size, len(rec['paths']), b+1, n_batches)
        # SR56_DISPLAY_END
        shortpep_monitor_update('RFD3_PROGRESS',completed_batches=b+1,requested_batches=n_batches)
finally:
    _release_engines()
_resume.save_stage(CKPT_FILE,'rfd3',{'cfg':_rfd3_fp,'module_cfg':_rfd3_fp,
                   'n_requested':len(all_designs),'n_generated':sum(a is not None for _,a,_ in all_designs)},
                   files=sorted(RFD3_DIR.glob(f'rfd3_{pdb_tag}_B*_D*.cif')))
print(f'✅ RFD3 请求 {len(all_designs)}；实际输出 {sum(a is not None for _,a,_ in all_designs)}；原编号保留')
# SR56_FEEDBACK_BEGIN
_feedback_safe(lambda: _feedback.event('END', 'main_rfd3', status='COMPLETED'))
# SR56_FEEDBACK_END


In [ ]:
# 【固定流程 · 无需修改】读取上方参数区的预览编号。
# ── 🎛️ 预览编号在上方「设计参数」单元直接修改 ──

In [ ]:
# 【固定流程 · 无需修改】保持骨架编号与下游范围；无效骨架保留空占位。
# ── 原预览位置与下游范围；只检查数据有效性 ──
n_designs = min(max_designs,len(all_designs)) if max_designs else len(all_designs)
designs_to_process = all_designs[:n_designs]
_backbone_validation = {}
_backbone_rows = []
for di,(name,aa,nr) in enumerate(all_designs):
    check = _validate_complex(aa)
    _backbone_validation[di] = check
    if check['valid_output']:
        check = _assess_model(aa,aa,{})
    _backbone_rows.append({'design_index':di,'design_name':name,'selected_for_downstream':di<n_designs,
                           'valid_output':check['valid_output'],'errors':'|'.join(check['errors']),
                           **{k:_metric_csv(v) for k,v in check['metrics'].items()}})
_valid_design_indices = [i for i in range(n_designs) if _backbone_validation[i]['valid_output']]
with open(MAIN_REPORT_DIR/'backbone_validity_audit.csv','w',newline='') as f:
    w = _csv.DictWriter(f,fieldnames=['design_index','design_name','selected_for_downstream','valid_output','errors',*QC_METRIC_FIELDS])
    w.writeheader(); w.writerows(_backbone_rows)
print(f'数据有效骨架 {len(_valid_design_indices)}/{n_designs}；指标无数值淘汰门槛')
_preview_id = min(preview_batch*diffusion_batch_size+preview_in_batch,max(0,len(all_designs)-1))
if all_designs and all_designs[_preview_id][1] is not None:
    d_name, gen_aa, n_res = all_designs[_preview_id]
    print('📷 骨架预览:',d_name,n_res,'残基')
else:
    gen_aa = None
    print('无可预览骨架；保留空结果路径')
if not _valid_design_indices: print('NO_VALID_BACKBONES：下游引擎不初始化')


In [ ]:
# 【固定流程 · 无需修改】展示生成骨架；本阶段无 pLDDT。
# ── 3D 预览（保留原单元位置）──
if gen_aa is None:
    print('无可预览骨架；继续空结果路径')
else:
    print('📷 设计 3D 预览')
    display(view(gen_aa))


---
## 🧪 第 2 步：MPNN 序列设计

固定同一 32 aa 靶标链，仅对有效骨架设计 40–60 aa binder 序列。原设计编号不压缩；无效骨架保留空序列占位。


In [ ]:
# 【固定流程 · 无需修改】整理输入和恢复辅助函数。
from pathlib import Path
# 权重路径由环境单元统一读取 MPNN_CKPT；不覆盖参数区的 model_type。
if _valid_design_indices and not Path(MPNN_CKPT).exists():
    raise RuntimeError(f'MPNN checkpoint 不存在: {MPNN_CKPT}；默认需要 proteinmpnn_v_48_020.pt')
print('MPNN model:',mpnn_model)
print('MPNN ckpt :',MPNN_CKPT)


In [ ]:
# 【固定流程 · 无需修改】执行主阶段 MPNN，保存复合物与序列。
# —— 第 2 步：MPNN — 固定 helix32，设计 minibinder 序列 ——
from mpnn.inference_engines.mpnn import MPNNInferenceEngine
print('第 2 步：MPNN 序列设计')
t0 = time.time()
_release_engines()
_mpnn_fp = _module_fp('mpnn')
all_mpnn_results = []

# SR56_DISPLAY_BEGIN
_sr56_tick('MPNN', False, 0, 0, reset=True)
# SR56_DISPLAY_END
try:
    for di,(name,aa,nr) in enumerate(designs_to_process):
        if di not in _valid_design_indices:
            # SR56_FEEDBACK_BEGIN
            _feedback_safe(lambda: _feedback.event('INVALID','main_mpnn',key=f'mpnn_design_{di}',design_index=di,errors=['backbone_invalid']))
            # SR56_FEEDBACK_END
            all_mpnn_results.append((name,aa,[])); continue
        b,i = divmod(di,diffusion_batch_size)
        seed = _stage_seed('main','mpnn',di)
        key = f'mpnn_design_{di}'
        fp = _config_fingerprint({'cfg':_mpnn_fp,'design':di,'seed':seed})
        # SR56_FEEDBACK_BEGIN
        _feedback_safe(lambda: _feedback.event('START', 'main_mpnn', key=key))
        # SR56_FEEDBACK_END
        rec = _stage_record(CKPT_FILE,key,fp)

        # SR56_DISPLAY_BEGIN
        _sr56_display_reused = rec is not None
        # SR56_DISPLAY_END
        # SR56_FEEDBACK_BEGIN
        _feedback_safe(lambda: _feedback.event('REUSED', 'main_mpnn', key=key, **rec)) if rec is not None else None
        # SR56_FEEDBACK_END
        if rec is None:
            _require_cuda()
            if mpnn is None:
                mpnn = MPNNInferenceEngine(model_type=mpnn_model,checkpoint_path=MPNN_CKPT,
                    is_legacy_weights=True,out_directory=None,write_structures=False,write_fasta=False)
            _seed_engine(mpnn,seed)
            results = mpnn.run(input_dicts=[_fix.refine_mpnn_input_dict(aa,input_aa,batch_size=mpnn_seqs_per_backbone)],
                               atom_arrays=[_fix.prepare_atom_array_for_mpnn(aa)])
            results = results or []
            if len(results)>mpnn_seqs_per_backbone: raise RuntimeError('MPNN 返回序列数量超过预算')
            paths = []
            for si,out in enumerate(results):
                _check_mpnn_backbone(aa,out.atom_array)
                p = MPNN_DIR/f'mpnn_{pdb_tag}_B{b}_D{i}_S{si}.cif'
                to_cif_file(out.atom_array,str(p)); paths.append(str(p))
            rec = {'cfg':fp,'seed':seed,'paths':paths,'n_requested':mpnn_seqs_per_backbone,'n_generated':len(paths)}
            _resume.save_stage(CKPT_FILE,key,rec,files=paths)
            # SR56_FEEDBACK_BEGIN
            _feedback_safe(lambda: _feedback.event('COMMITTED', 'main_mpnn', key=key, **rec))
            # SR56_FEEDBACK_END
        seqs = [SimpleNamespace(atom_array=_read_cif(p)) for p in rec['paths']]
        for out in seqs: _check_mpnn_backbone(aa,out.atom_array)
        all_mpnn_results.append((name,aa,seqs))
        with open(MPNN_DIR/f'mpnn_{pdb_tag}_B{b}_D{i}.fasta','w') as f:
            for si,out in enumerate(seqs): f.write(f'>{pdb_tag}_B{b}_D{i}_S{si}\n{_binder_seq_of(out.atom_array)}\n')
        print(f'[{di}] {len(seqs)}/{mpnn_seqs_per_backbone} 条序列')
        # SR56_DISPLAY_BEGIN
        _sr56_tick('MPNN', _sr56_display_reused, mpnn_seqs_per_backbone, len(seqs), di+1, len(designs_to_process))
        # SR56_DISPLAY_END

finally:
    _release_engines()
assert len(all_mpnn_results)==len(designs_to_process)
assert all(name==designs_to_process[i][0] for i,(name,aa,seqs) in enumerate(all_mpnn_results))
assert all(not all_mpnn_results[i][2] for i in range(len(all_mpnn_results)) if i not in _valid_design_indices)
total_seqs = sum(len(r[2]) for r in all_mpnn_results)
_resume.save_stage(CKPT_FILE,'mpnn',{'cfg':_mpnn_fp,'module_cfg':_mpnn_fp,'n_designs':len(all_mpnn_results),
                   'n_valid_backbones':len(_valid_design_indices),'n_sequences':total_seqs},
                   files=sorted(MPNN_DIR.glob(f'mpnn_{pdb_tag}_B*_D*_S*.cif')))
print(f'✅ MPNN {total_seqs} 条；耗时 {time.time()-t0:.1f}s')
# SR56_FEEDBACK_BEGIN
_feedback_safe(lambda: _feedback.event('END', 'main_mpnn', status='COMPLETED'))
# SR56_FEEDBACK_END

# SR56_DISPLAY_BEGIN
_sr56_table('MPNN 完整序列清单', ({'design_index':di, 'sequence_index':si, 'sequence':_binder_seq_of(out.atom_array)} for di,(_name,_aa,_seqs) in enumerate(all_mpnn_results) for si,out in enumerate(_seqs)), [('design_index','设计'),('sequence_index','序列编号'),('sequence','完整 binder 序列')], MAIN_REPORT_DIR/'mpnn_sequence_display.csv', '完整序列与原阶段 FASTA 对应；显示表不参与筛选。')
# SR56_DISPLAY_END


---
## 🔬 第 3 步：RF3 结构验证

以同一 32 aa 靶标和 MPNN binder 序列组成纯序列复合物输入，无坐标模板。逐候选保存全部 RF3 模型；数据有效性决定是否可参与排名，置信度、接触、碰撞和 RMSD 只报告。


In [ ]:
# 【固定流程 · 无需修改】执行主阶段 RF3，保存实际返回模型及逐模型评估。
# —— 第 3 步：RF3 — 同一 helix32 + minibinder 的纯序列复合物 ——
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput
print('第 3 步：RF3 结构验证')
t0 = time.time()
_release_engines()

def _rf3_seq_only(aa,eid):
    return InferenceInput.from_json_dict(_complex_components(aa,eid),
        template_selection=[],ground_truth_conformer_selection=[])

def _restore_models(records):
    return [(r['mi'],SimpleNamespace(atom_array=_read_cif(r['cif']),
            summary_confidences=json.loads(Path(r['conf']).read_text()).get('summary',{}),
            confidences=json.loads(Path(r['conf']).read_text()).get('confidences',{}))) for r in records]

all_rf3_results = []
_all_models_audit = []
_pair_rows = []
_best_model_indices = {}
_RF3_MODULE_FP = _module_fp('rf3')
_RF3_PERSIST = RF3_DIR
_RF3_PERSIST.mkdir(parents=True,exist_ok=True)
_RF3_CKPT = MAIN_RF3_CHECKPOINT_DIR/'complete.json'
total_jobs = 0

# SR56_DISPLAY_BEGIN
_sr56_tick('RF3', False, 0, 0, reset=True)
_sr56_display_total = sum(len(seqs) if rf3_validations==0 else min(rf3_validations,len(seqs)) for _name,_aa,seqs in all_mpnn_results)
# SR56_DISPLAY_END
_pair_checkpoints = []
try:
    for di,(name,backbone,seqs) in enumerate(all_mpnn_results):
        n_val = len(seqs) if rf3_validations==0 else min(rf3_validations,len(seqs))
        for si,seqout in enumerate(seqs[:n_val]):
            aa = seqout.atom_array
            check = _validate_complex(aa)
            if not check['valid_output']:
                # SR56_FEEDBACK_BEGIN
                _feedback_safe(lambda: _feedback.event('INVALID','main_rf3',key=f'main:D{di}:S{si}',design_index=di,sequence_index=si,round=0,errors=check['errors']))
                # SR56_FEEDBACK_END
                _pair_rows.append({'design_index':di,'sequence_index':si,'n_generated':0,'n_valid':0,
                    'n_invalid':0,'valid_fraction':'','best_model_index':'','status':'invalid_input',
                    'errors':'|'.join(check['errors'])})
                continue
            seed = _stage_seed('main','rf3',di,si)
            input_seq = _binder_seq_of(aa)
            key = f'rf3_{di}_{si}'
            pair_dir = _RF3_PERSIST/f'd{di}_s{si}'
            pair_ckpt = MAIN_RF3_CHECKPOINT_DIR/f'd{di}_s{si}'/'complete.json'
            b,i = divmod(di,diffusion_batch_size)
            input_file = MPNN_DIR/f'mpnn_{pdb_tag}_B{b}_D{i}_S{si}.cif'
            fp = _config_fingerprint({'cfg':_RF3_MODULE_FP,'input':_sha256_file(input_file),'seed':seed})
            # SR56_FEEDBACK_BEGIN
            _feedback_safe(lambda: _feedback.event('START', 'main_rf3', key=key))
            # SR56_FEEDBACK_END
            rec = _stage_record(pair_ckpt,key,fp)

            # SR56_DISPLAY_BEGIN
            _sr56_display_reused = rec is not None
            # SR56_DISPLAY_END
            # SR56_FEEDBACK_BEGIN
            _feedback_safe(lambda: _feedback.event('REUSED', 'main_rf3', key=key, **rec)) if rec is not None else None
            # SR56_FEEDBACK_END
            if rec is None:
                _require_cuda()
                if rf3_engine is None:
                    _quiet_logging()
                    rf3_engine = RF3InferenceEngine(ckpt_path=RF3_CKPT,verbose=False,
                                                   diffusion_batch_size=rf3_models_per_sequence)
                _seed_engine(rf3_engine,seed)
                eid = f'd{di}_s{si}'
                outputs = rf3_engine.run(inputs=_rf3_seq_only(aa,eid),annotate_b_factor_with_plddt=True)
                raw = (outputs or {}).get(eid,[])
                records = []
                for mi,out in enumerate(raw):
                    cifp,confp = _resume.save_rf3_output(pair_dir,di,si,out,model_idx=mi)
                    records.append({'mi':mi,'cif':cifp,'conf':confp})
                rec = {'cfg':fp,'seed':seed,'input_seq':input_seq,'models':records,
                       'n_requested_models':rf3_models_per_sequence,'n_generated_models':len(records)}
                _resume.save_stage(pair_ckpt,key,rec,files=[p for r in records for p in (r['cif'],r['conf'])])
                # SR56_FEEDBACK_BEGIN
                _feedback_safe(lambda: _feedback.event('COMMITTED', 'main_rf3', key=key, **rec))
                # SR56_FEEDBACK_END
            _pair_checkpoints.append(pair_ckpt)
            if rec['input_seq']!=input_seq: raise RuntimeError('RF3 断点序列身份不符')
            models = _restore_models(rec['models'])
            total_jobs += 1

            # SR56_DISPLAY_BEGIN
            _sr56_tick('RF3', _sr56_display_reused, rf3_models_per_sequence, len(models), total_jobs, _sr56_display_total)
            # SR56_DISPLAY_END
            pool = []
            for mi,out in models:
                check = _assess_model(aa,out.atom_array,out.summary_confidences,expected_binder_seq=input_seq)
                _all_models_audit.append((di,si,mi,check['valid_output'],check['errors'],check['metrics']))
                # SR56_FEEDBACK_BEGIN
                _feedback_safe(lambda: _feedback_model(rec,mi,out,check,di,si,RFD3_DIR/f'rfd3_{pdb_tag}_B{b}_D{i}.cif',input_file))
                # SR56_FEEDBACK_END
                if check['valid_output']: pool.append((mi,out))
            selected = max(pool,key=lambda t:_rank_key(t[1].summary_confidences,(t[0],))) if pool else None
            # SR56_FEEDBACK_BEGIN
            _feedback_safe(lambda: _feedback_pair(rec,selected,di,si,RFD3_DIR/f'rfd3_{pdb_tag}_B{b}_D{i}.cif',input_file))
            # SR56_FEEDBACK_END
            _pair_rows.append({'design_index':di,'sequence_index':si,'n_generated':len(models),'n_valid':len(pool),
                'n_invalid':len(models)-len(pool),'valid_fraction':len(pool)/len(models) if models else '',
                'best_model_index':selected[0] if selected else '',
                'status':'valid_prediction_output' if selected else ('no_output' if not models else 'invalid_outputs'),
                'errors':'' if selected else '|'.join(sorted({e for r in _all_models_audit if r[:2]==(di,si) for e in r[4]}))})
            if selected:
                mi,out = selected
                _best_model_indices[(di,si)] = mi
                all_rf3_results.append((name,di,si,input_seq[:30],_report_summary(out.summary_confidences),out))
finally:
    _release_engines()
all_rf3_results.sort(key=lambda x:_rank_key(x[4],(x[1],x[2],_best_model_indices[(x[1],x[2])])),reverse=True)
with open(MAIN_REPORT_DIR/'rf3_model_audit.csv','w',newline='') as f:
    w = _csv.writer(f); w.writerow(['design_index','sequence_index','model_index','valid_output','errors','validation_version',*QC_METRIC_FIELDS])
    for di,si,mi,valid,errors,metrics in _all_models_audit:
        w.writerow([di,si,mi,valid,'|'.join(errors),VALIDATION_VERSION,*[_metric_csv(metrics[k]) for k in QC_METRIC_FIELDS]])
with open(MAIN_REPORT_DIR/'rf3_pair_summary.csv','w',newline='') as f:
    w = _csv.DictWriter(f,fieldnames=['design_index','sequence_index','n_generated','n_valid','n_invalid','valid_fraction','best_model_index','status','errors'])
    w.writeheader(); w.writerows(_pair_rows)
_stage_counts = {'requested_backbones':len(all_designs),'generated_backbones':sum(a is not None for _,a,_ in all_designs),
                 'selected_backbones':len(designs_to_process),'valid_backbones':len(_valid_design_indices),
                 'mpnn_sequences':sum(len(r[2]) for r in all_mpnn_results),'rf3_pairs_considered':len(_pair_rows),
                 'rf3_pairs_validated_or_reused':total_jobs,'rf3_models':len(_all_models_audit),
                 'rf3_models_valid':sum(r[3] for r in _all_models_audit),'rf3_pairs_with_valid_outputs':len(all_rf3_results),
                 'status':'VALID_PREDICTION_OUTPUTS' if all_rf3_results else 'NO_VALID_PREDICTION_OUTPUTS',
                 'validation_version':VALIDATION_VERSION}
_write_json_atomic(MAIN_REPORT_DIR/'stage_counts.json',_stage_counts)
_resume.save_stage(_RF3_CKPT,'main_rf3_summary',{'cfg':_RF3_MODULE_FP,**_stage_counts},files=_pair_checkpoints)
_resume.save_stage(CKPT_FILE,'rf3',{'cfg':_RF3_MODULE_FP,'n':len(all_rf3_results),'persist':str(_RF3_PERSIST)},
                   files=[_RF3_CKPT] if _RF3_CKPT.exists() else [])
print('完整阶段计数:',_stage_counts)
print(f'✅ 有效预测候选 {len(all_rf3_results)}；指标只报告；耗时 {time.time()-t0:.1f}s')
# SR56_FEEDBACK_BEGIN
_feedback_safe(lambda: _feedback.event('END', 'main_rf3', status='COMPLETED'))
# SR56_FEEDBACK_END

# SR56_DISPLAY_BEGIN
_sr56_table('RF3 每条序列最佳实际模型 · 完整序列', ({'rank':rank,'design_index':di,'sequence_index':si,'model_index':_best_model_indices[(di,si)],'sequence':_binder_seq_of(all_mpnn_results[di][2][si].atom_array),**summary} for rank,(_name,di,si,_short,summary,_out) in enumerate(all_rf3_results,1)), [('rank', '原榜名次'), ('design_index', '设计'), ('sequence_index', '序列编号'), ('model_index', '模型编号'), ('ranking_score', 'ranking'), ('overall_plddt', 'pLDDT'), ('ptm', 'pTM'), ('iptm', 'ipTM'), ('overall_pae', 'PAE'), ('sequence', '完整 binder 序列')], MAIN_RANKING_DIR/'rf3_sequence_display.csv', '原生排序及原生最佳模型编号保持；无新增接受门槛。')
# SR56_DISPLAY_END


In [ ]:
# 【固定流程 · 无需修改】展示主阶段最佳模型的指标与结构。
# ── 最佳结构详细报告（原位置）──
if not all_rf3_results:
    print('没有有效 RF3 输出；跳过最佳结构展示，继续空表导出')
else:
    best_dname,best_didx,best_sidx,best_seq,best_summary,best_output = all_rf3_results[0]
    best_mi = _best_model_indices[(best_didx,best_sidx)]
    print(f'📷 最佳有效预测 D{best_didx}:S{best_sidx}:M{best_mi}')
    with _fold_out('📊 最佳结构置信度详细报告'):
        print('原生摘要:',best_output.summary_confidences)
        print('该候选模型数/有效数:',next((r['n_generated'],r['n_valid']) for r in _pair_rows if r['design_index']==best_didx and r['sequence_index']==best_sidx))
        print('诊断:',_assess_model(all_mpnn_results[best_didx][2][best_sidx].atom_array,best_output.atom_array,best_summary)['metrics'])

    # SR56_DISPLAY_BEGIN
    _sr56_display_current = _sr56_current_best()
    if _sr56_display_current is not None:
        _sr56_table('当前 RF3 最佳模型值与完整序列', [_sr56_display_current['table']], [('rank', '原榜名次'), ('design_index', '设计'), ('sequence_index', '序列编号'), ('model_index', '模型编号'), ('ranking_score', 'ranking'), ('overall_plddt', 'pLDDT'), ('ptm', 'pTM'), ('iptm', 'ipTM'), ('overall_pae', 'PAE'), ('binder_self_rmsd_A', 'binder self RMSD Å'), ('binder_pose_rmsd_A', 'binder pose RMSD Å'), ('target_fragment_rmsd_A', 'target RMSD Å'), ('sequence', '完整 binder 序列')], MAIN_RANKING_DIR/'rf3_current_best_display.csv', '本单元读取当前 RF3 输出，不读取尚未生成的主设计排名。')
        _sr56_three_panel('当前 RF3 RFD3 / MPNN / RF3 三栏', [_sr56_display_current['view']], expanded=True)
    # SR56_DISPLAY_END


---
## 📏 第 4 步：度量与综合排名

区分 binder self-RMSD、32 aa 靶标参考系 pose-RMSD 和 target fragment RMSD。每个骨架设计选最高排名的有效预测，缺失排名保留；不使用数值阈值排除候选。


In [ ]:
# 【固定流程 · 无需修改】按既定 ranking_score 规则生成主阶段正式排名。
# —— 第 4 步：度量与每骨架设计综合排名 ——
print('第 4 步：片段度量与综合排名；无 RMSD 淘汰门槛')

def rmsd_label(value,thresholds,labels):
    for threshold,label in zip(thresholds,labels):
        if value<threshold: return label
    return labels[-1]

candidate_rows = []
design_cand = {}
for name,di,si,head,summary,out in all_rf3_results:
    backbone = designs_to_process[di][1]
    mpnn_aa = all_mpnn_results[di][2][si].atom_array
    check = _assess_model(mpnn_aa,out.atom_array,summary,expected_binder_seq=_binder_seq_of(mpnn_aa))
    mi = _best_model_indices[(di,si)]
    candidate = {'d_idx':di,'d_name':name,'s_idx':si,'model_index':mi,'summary':summary,
                 'rf3_out':out,'orig_aa':backbone,'mpnn_aa':mpnn_aa,'metrics':check['metrics'],
                 'valid_output':check['valid_output'],'errors':check['errors'],
                 'm_rmsd':check['metrics']['target_fragment_rmsd_A'],
                 'pose':check['metrics']['binder_pose_rmsd_A'],'self_r':check['metrics']['binder_self_rmsd_A']}
    candidate_rows.append(candidate)
    if check['valid_output'] and (di not in design_cand or
        _rank_key(summary,(si,mi))>_rank_key(design_cand[di]['summary'],(design_cand[di]['s_idx'],design_cand[di]['model_index']))):
        design_cand[di] = candidate
design_best = {di:(c['d_name'],c['orig_aa'],c['rf3_out'],c['summary'],c['s_idx'],c['mpnn_aa']) for di,c in design_cand.items()}
anchor_report = {di:({'fragment32':c['m_rmsd']},_interface_stats(c['rf3_out'].atom_array)) for di,c in design_cand.items()}
rmsd_report = [(di,c['d_name'],c['summary'],c['m_rmsd'],c['pose'],c['s_idx']) for di,c in design_cand.items()]
rmsd_report.sort(key=lambda r:_rank_key(r[2],(r[0],r[5],design_cand[r[0]]['model_index'])),reverse=True)
_ranking_html = []
for rank,(di,name,summary,fragment,pose,si) in enumerate(rmsd_report,1):
    c = design_cand[di]
    _ranking_html.append(f'<tr><td>{rank}</td><td>D{di}:S{si}:M{c["model_index"]}</td>'
                         f'<td>{summary["ranking_score"]:.3f}</td><td>{fragment:.3f}</td><td>{pose:.3f}</td>'
                         f'<td>{c["self_r"]:.3f}</td><td>{c["metrics"]["contact_residue_ids"]}</td></tr>')
display(_collapsible(f'🏆 每骨架设计排名（{len(rmsd_report)} 个有效预测）',
    '<table><tr><th>排名</th><th>设计</th><th>ranking</th><th>片段 RMSD</th><th>pose RMSD</th><th>self RMSD</th><th>接触位置</th></tr>'
    +''.join(_ranking_html)+'</table>'))
if not rmsd_report: print('NO_VALID_PREDICTION_OUTPUTS：空结果；不补入历史候选')
# SR56_FEEDBACK_BEGIN
for _feedback_rank, (_feedback_di, _feedback_name, _feedback_summary, _feedback_fragment, _feedback_pose, _feedback_si) in enumerate(rmsd_report, 1):
    _feedback_safe(lambda: _feedback.event('FINAL_SELECTED', 'main_rf3', key=_feedback_id(_feedback_di, _feedback_si, design_cand[_feedback_di]['model_index']), id=_feedback_id(_feedback_di, _feedback_si, design_cand[_feedback_di]['model_index']), design_index=_feedback_di, round=0, formal_rank=_feedback_rank, selection='main'))
# SR56_FEEDBACK_END


In [ ]:
# 【固定流程 · 无需修改】导出主阶段排名、序列与结构。
# ── 导出排名前 5（原位置）；全排名表保留全部骨架设计 ──
import csv
n_export = min(5,len(rmsd_report))
report_path = MAIN_RANKING_DIR/'ranking_report.csv'
with open(report_path,'w',newline='') as f:
    fields = ['rank','design_index','sequence_index','model_index','ranking_score','overall_plddt','ptm','iptm',
              'overall_pae','RF3_has_clash',*QC_METRIC_FIELDS]
    w = csv.DictWriter(f,fieldnames=fields); w.writeheader()
    for rank,(di,name,summary,fragment,pose,si) in enumerate(rmsd_report,1):
        c = design_cand[di]
        row = {'rank':rank,'design_index':di,'sequence_index':si,'model_index':c['model_index'],
               **{k:_metric_csv(summary.get(k)) for k in ('ranking_score','overall_plddt','ptm','iptm','overall_pae')},
               'RF3_has_clash':summary.get('has_clash',''),**{k:_metric_csv(c['metrics'][k]) for k in QC_METRIC_FIELDS}}
        w.writerow(row)
with open(RF3_DIR/'top_design_binders.fasta','w') as f:
    for rank,(di,name,summary,fragment,pose,si) in enumerate(rmsd_report[:n_export],1):
        c = design_cand[di]; out = c['rf3_out']; b,i = divmod(di,diffusion_batch_size)
        stem = f'rf3_{pdb_tag}_B{b}_D{i}_S{si}_M{c["model_index"]}'
        to_cif_file(c['orig_aa'],str(RF3_DIR/(stem+'_generated.cif')))
        to_cif_file(c['mpnn_aa'],str(RF3_DIR/(stem+'_mpnn.cif')))
        to_cif_file(out.atom_array,str(RF3_DIR/(stem+'_refolded.cif')))
        to_cif_file(_extract_binder(out.atom_array),str(RF3_DIR/(stem+'_binder.cif')))
        f.write(f'>{stem} ranking={_metric_csv(summary["ranking_score"])}\n{_binder_seq_of(out.atom_array)}\n')
        print(f'排名 {rank}: {stem} | 接触 {c["metrics"]["contact_residue_ids"]}')
print('📊 排名 CSV:',report_path)
print('📁 实际导出',n_export,'个设计')
# 原导出区域仍展示生成结构与重折叠结构。

# SR56_DISPLAY_BEGIN
_sr56_table('主阶段正式榜 · 完整序列与三种 RMSD', _sr56_main_rows(), [('rank', '原榜名次'), ('design_index', '设计'), ('sequence_index', '序列编号'), ('model_index', '模型编号'), ('ranking_score', 'ranking'), ('overall_plddt', 'pLDDT'), ('ptm', 'pTM'), ('iptm', 'ipTM'), ('overall_pae', 'PAE'), ('binder_self_rmsd_A', 'binder self RMSD Å'), ('binder_pose_rmsd_A', 'binder pose RMSD Å'), ('target_fragment_rmsd_A', 'target RMSD Å'), ('sequence', '完整 binder 序列')], MAIN_RANKING_DIR/'main_selected_display.csv', '完整正式榜仍见 ranking_report.csv。')
_sr56_three_panel('主阶段多设计三栏（选择切换，展开后才创建视图）', _sr56_main_views(n_export))
# SR56_DISPLAY_END


---
## 🔄 循环 refine

前 10 个骨架设计运行 3 轮部分扩散、MPNN 和 RF3。每轮同一 32 aa 靶标固定，每父路径 8 骨架 × 8 序列，后续最多 3 个不同来源骨架路径。第 0 轮 baseline 始终保留；导出全循环最佳结果。

按阶段释放引擎并记录断点，支持恢复部分完成的轮次；不添加碰撞、接触、RMSD 或置信度门槛。


In [ ]:
# 【固定流程 · 无需修改】推导 refine 工作量与路径；参数在上方研究参数区修改。
# ══ refine 参数保持在上方原参数区；本单元只计算派生值 ══
refine_partial_t = np.linspace(refine_partial_t_start,refine_partial_t_end,refine_rounds)
refine_n_backbones = refine_backbone_batch*refine_batch
_refine_cfg_fp = _config_fingerprint({'main':_module_fp('rf3'),'validation':VALIDATION_VERSION,
    'refine':{k:globals()[k] for k in ('refine_top_n','refine_rounds','refine_partial_t_start','refine_partial_t_end',
              'refine_timesteps','refine_mpnn_seqs','refine_rf3_validations','refine_backbone_batch','refine_batch','refine_beam_k')},
    'report_distances':[report_contact_distance_angstrom,report_clash_distance_angstrom]})
REFINE_DIR = REPORT_DIR/'refine'
REFINE_RFD3_DIR, REFINE_MPNN_DIR, REFINE_RF3_DIR = (OUT_DIR/'refine'/name for name in ('rfd3','mpnn','rf3'))
for d in (REFINE_DIR,REFINE_RFD3_DIR,REFINE_MPNN_DIR,REFINE_RF3_DIR): d.mkdir(parents=True,exist_ok=True)
_REFINE_CKPT = CHECKPOINT_DIR/'refine/refine_complete.json'
n_top = min(refine_top_n,len(rmsd_report))
t0 = time.time()
print(f'🔄 前 {n_top} 个骨架设计 × {refine_rounds} 轮 refine；第 0 轮 baseline 保留')


In [ ]:
# 【固定流程 · 无需修改】执行三轮 refine，逐步骤恢复并保留 baseline 与全循环最佳。
# ———循环 refine：每轮分阶段运行，逐昂贵步骤恢复————
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
from mpnn.inference_engines.mpnn import MPNNInferenceEngine
from rf3.inference_engines.rf3 import RF3InferenceEngine

_release_engines()
refine_history = {}; refine_best = {}; refine_best_round = {}; refine_best_gen = {}
refine_best_mpnn = {}; refine_rankings = {}; refine_iptms = {}; beam_kept = {}; converged = {}
refine_best_model_index = {}
_refine_model_audit = []; _refine_backbone_audit = []; _refine_pair_rows = []
_all_candidate_rows = []

# SR56_DISPLAY_BEGIN
_sr56_display_retained = []
# SR56_DISPLAY_END
_refine_counts = {'requested_rfd3_backbones':0,'generated_rfd3_backbones':0,'valid_backbones':0,
                  'mpnn_sequences':0,'rf3_pairs_validated_or_reused':0,'rf3_models':0,'rf3_valid_models':0,
                  'designs_requested':n_top,'rounds_completed':0}

try:
    for rank_idx,(di,name,s0,fragment0,pose0,si0) in enumerate(rmsd_report[:n_top]):
        baseline = design_best[di][2]
        baseline_mpnn = design_best[di][5]
        orig_aa = designs_to_process[di][1]
        mi0 = design_cand[di]['model_index']
        check0 = _assess_model(baseline_mpnn,baseline.atom_array,s0,expected_binder_seq=_binder_seq_of(baseline_mpnn))
        if not check0['valid_output']: raise RuntimeError('refine baseline data changed')
        refine_history[di] = [(0,pose0,s0['overall_plddt'],s0['ptm'],-1,si0,f'P{rank_idx}',check0['metrics']['binder_self_rmsd_A'],fragment0)]
        refine_rankings[di] = {0:s0['ranking_score']}; refine_iptms[di] = {0:s0['iptm']}
        refine_best_round[di] = (0,-1,si0,baseline.atom_array,s0,baseline,f'P{rank_idx}')
        refine_best_gen[di] = orig_aa; refine_best_mpnn[di] = baseline_mpnn
        refine_best_model_index[di] = mi0
        beam_kept[di] = {}; converged[di] = False
        best_key = _rank_key(s0,(0,0,0,si0,mi0))
        last_out = baseline; current_aa = baseline.atom_array; last_summary = s0
        last_round,last_k,last_s = 0,-1,si0
        beam = [(0,_reset_fragment_target(baseline.atom_array),f'P{rank_idx}')]
        # SR56_FEEDBACK_BEGIN
        _feedback_safe(lambda: _feedback_baseline(di,si0,mi0,f'P{rank_idx}'))
        # SR56_FEEDBACK_END
        design_fp = _config_fingerprint({'cfg':_refine_cfg_fp,'design':di,'sequence':si0,'model':mi0,
                        'baseline_sequence':_binder_seq_of(baseline.atom_array),
                        'baseline_coords':baseline.atom_array.coord.tolist()})

        for rnd in range(1,refine_rounds+1):
            print(f'🔄 D{di} C{rnd}: {len(beam)} 父路径')
            if len(beam)>refine_beam_k: raise RuntimeError('refine beam 超过预算')
            round_fp = _config_fingerprint({'design':design_fp,'round':rnd,
                               'parents':[(pi,path,aa.coord.tolist()) for pi,aa,path in beam]})
            rfd_backbones = []

            # 1. RFD3: all parents for this round, then release its engine.
            _release_engines()
            try:
                for pi,parent_aa,parent_path in beam:
                    seed = _stage_seed('refine','rfd3',di,rnd,pi,parent_path)
                    key = f'D{di}_C{rnd}_P{pi}_rfd3'
                    fp = _config_fingerprint({'round':round_fp,'seed':seed,'stage':'rfd3'})
                    # SR56_FEEDBACK_BEGIN
                    _feedback_safe(lambda: _feedback.event('START', 'refine_rfd3', key=key))
                    # SR56_FEEDBACK_END
                    rec = _stage_record(_REFINE_CKPT,key,fp)
                    # SR56_FEEDBACK_BEGIN
                    _feedback_safe(lambda: _feedback.event('REUSED', 'refine_rfd3', key=key, **rec)) if rec is not None else None
                    # SR56_FEEDBACK_END
                    if rec is None:
                        _require_cuda()
                        if rfd3_model is None:
                            rfd3_model = RFD3InferenceEngine(**RFD3InferenceConfig(
                                ckpt_path=RFD3_CKPT,diffusion_batch_size=refine_backbone_batch,
                                low_memory_mode=low_memory_mode,
                                inference_sampler={'num_timesteps':refine_timesteps,'step_scale':step_scale,'gamma_0':gamma_0}))
                        spec,tmp_cif,tmp_json = _fix.build_partial_diffusion_spec(
                            parent_aa,float(refine_partial_t[rnd-1]),fixed_chains=[_resolve_target_chain(parent_aa)],
                            redesign_sequences=False,is_non_loopy=is_non_loopy)
                        try:
                            _seed_engine(rfd3_model,seed)
                            outs = rfd3_model.run(inputs=tmp_json,out_dir=None,n_batches=refine_batch)
                        finally:
                            os.unlink(tmp_cif); os.unlink(tmp_json); os.rmdir(str(Path(tmp_cif).parent))
                        raw = [out for group in (outs or {}).values() for out in group]
                        if len(raw)>refine_n_backbones: raise RuntimeError('refine RFD3 输出超过预算')
                        paths = []
                        for k,out in enumerate(raw):
                            p = REFINE_RFD3_DIR/f'rfd3_refine_{pdb_tag}_D{di}_C{rnd}_P{pi}_K{k}.cif'
                            to_cif_file(out.atom_array,str(p)); paths.append(str(p))
                        rec = {'cfg':fp,'seed':seed,'parent_path':parent_path,'paths':paths,
                               'n_requested':refine_n_backbones,'n_generated':len(paths)}
                        _resume.save_stage(_REFINE_CKPT,key,rec,files=paths)
                        # SR56_FEEDBACK_BEGIN
                        _feedback_safe(lambda: _feedback.event('COMMITTED', 'refine_rfd3', key=key, **rec))
                        # SR56_FEEDBACK_END
                    _refine_counts['requested_rfd3_backbones'] += refine_n_backbones
                    _refine_counts['generated_rfd3_backbones'] += len(rec['paths'])
                    for k in range(refine_n_backbones):
                        aa = _read_cif(rec['paths'][k]) if k<len(rec['paths']) else None
                        check = _validate_complex(aa)
                        _refine_backbone_audit.append({'design_index':di,'round':rnd,'parent_index':pi,
                            'path':parent_path,'backbone':k,'valid_output':check['valid_output'],'errors':'|'.join(check['errors'])})
                        if check['valid_output']:
                            _refine_counts['valid_backbones'] += 1
                            rfd_backbones.append((pi,parent_path,k,aa,rec['paths'][k]))
            finally:
                _release_engines()

            # 2. MPNN: fixed 32 aa target on every valid generated backbone.
            sequence_inputs = []
            try:
                for pi,parent_path,k,aa,backbone_file in rfd_backbones:
                    seed = _stage_seed('refine','mpnn',di,rnd,pi,k,parent_path)
                    key = f'D{di}_C{rnd}_P{pi}_K{k}_mpnn'
                    fp = _config_fingerprint({'round':round_fp,'backbone':_sha256_file(backbone_file),'seed':seed,'stage':'mpnn'})
                    # SR56_FEEDBACK_BEGIN
                    _feedback_safe(lambda: _feedback.event('START', 'refine_mpnn', key=key))
                    # SR56_FEEDBACK_END
                    rec = _stage_record(_REFINE_CKPT,key,fp)
                    # SR56_FEEDBACK_BEGIN
                    _feedback_safe(lambda: _feedback.event('REUSED', 'refine_mpnn', key=key, **rec)) if rec is not None else None
                    # SR56_FEEDBACK_END
                    if rec is None:
                        _require_cuda()
                        if mpnn_eng is None:
                            mpnn_eng = MPNNInferenceEngine(model_type=mpnn_model,checkpoint_path=MPNN_CKPT,
                                is_legacy_weights=True,out_directory=None,write_structures=False,write_fasta=False)
                        _seed_engine(mpnn_eng,seed)
                        outputs = mpnn_eng.run(input_dicts=[_fix.refine_mpnn_input_dict(aa,input_aa,batch_size=refine_mpnn_seqs)],
                                                atom_arrays=[_fix.prepare_atom_array_for_mpnn(aa)]) or []
                        if len(outputs)>refine_mpnn_seqs: raise RuntimeError('refine MPNN 输出超过预算')
                        paths = []
                        for si,out in enumerate(outputs):
                            _check_mpnn_backbone(aa,out.atom_array)
                            p = REFINE_MPNN_DIR/f'mpnn_refine_{pdb_tag}_D{di}_C{rnd}_P{pi}_K{k}_S{si}.cif'
                            to_cif_file(out.atom_array,str(p)); paths.append(str(p))
                        rec = {'cfg':fp,'seed':seed,'paths':paths,'n_requested':refine_mpnn_seqs,'n_generated':len(paths)}
                        _resume.save_stage(_REFINE_CKPT,key,rec,files=paths)
                        # SR56_FEEDBACK_BEGIN
                        _feedback_safe(lambda: _feedback.event('COMMITTED', 'refine_mpnn', key=key, **rec))
                        # SR56_FEEDBACK_END
                    _refine_counts['mpnn_sequences'] += len(rec['paths'])
                    n_val = len(rec['paths']) if refine_rf3_validations==0 else min(refine_rf3_validations,len(rec['paths']))
                    for si,p in enumerate(rec['paths'][:n_val]):
                        seq_aa = _read_cif(p); _check_mpnn_backbone(aa,seq_aa)
                        sequence_inputs.append((pi,parent_path,k,si,seq_aa,aa,p,backbone_file))
            finally:
                _release_engines()

            # 3. RF3: every pair has its own small atomic checkpoint (bounded JSON writes).
            all_candidates = []
            try:
                for pi,parent_path,k,si,seq_aa,backbone,seq_file,backbone_file in sequence_inputs:
                    data = _validate_complex(seq_aa)
                    pair_base = {'design_index':di,'round':rnd,'parent_index':pi,'path':parent_path,'backbone':k,'sequence_index':si}
                    if not data['valid_output']:
                        # SR56_FEEDBACK_BEGIN
                        _feedback_safe(lambda: _feedback.event('INVALID','refine_rf3',key=f'refine:D{di}:C{rnd}:P{pi}:K{k}:S{si}',**pair_base,errors=data['errors']))
                        # SR56_FEEDBACK_END
                        _refine_pair_rows.append({**pair_base,'n_generated':0,'n_valid':0,'n_invalid':0,
                            'valid_fraction':'','best_model_index':'','status':'invalid_input','errors':'|'.join(data['errors'])}); continue
                    seed = _stage_seed('refine','rf3',di,rnd,pi,k,si,parent_path)
                    key = f'D{di}_C{rnd}_P{pi}_K{k}_S{si}'
                    pair_dir = REFINE_RF3_DIR/key
                    pair_ckpt = REFINE_RF3_CHECKPOINT_DIR/key/'complete.json'
                    fp = _config_fingerprint({'round':round_fp,'input':_sha256_file(seq_file),'seed':seed,'stage':'rf3'})
                    # SR56_FEEDBACK_BEGIN
                    _feedback_safe(lambda: _feedback.event('START', 'refine_rf3', key=key))
                    # SR56_FEEDBACK_END
                    rec = _stage_record(pair_ckpt,'rf3',fp)
                    # SR56_FEEDBACK_BEGIN
                    _feedback_safe(lambda: _feedback.event('REUSED', 'refine_rf3', key=key, **rec)) if rec is not None else None
                    # SR56_FEEDBACK_END
                    input_seq = _binder_seq_of(seq_aa)
                    if rec is None:
                        _require_cuda()
                        if rf3_eng is None:
                            rf3_eng = RF3InferenceEngine(ckpt_path=RF3_CKPT,verbose=False,diffusion_batch_size=rf3_models_per_sequence)
                        _seed_engine(rf3_eng,seed)
                        eid = 'refine_'+key
                        outputs = rf3_eng.run(inputs=_rf3_seq_only(seq_aa,eid),annotate_b_factor_with_plddt=True)
                        raw = (outputs or {}).get(eid,[])
                        records = []
                        for mi,out in enumerate(raw):
                            cifp,confp = _resume.save_rf3_output(pair_dir,di,si,out,model_idx=mi)
                            records.append({'mi':mi,'cif':cifp,'conf':confp})
                        rec = {'cfg':fp,'seed':seed,'input_seq':input_seq,'models':records,
                               'n_requested_models':rf3_models_per_sequence,'n_generated_models':len(records)}
                        _resume.save_stage(pair_ckpt,'rf3',rec,files=[p for r in records for p in (r['cif'],r['conf'])])
                        # SR56_FEEDBACK_BEGIN
                        _feedback_safe(lambda: _feedback.event('COMMITTED', 'refine_rf3', key=key, **rec))
                        # SR56_FEEDBACK_END
                    if rec['input_seq']!=input_seq: raise RuntimeError('refine RF3 断点序列身份不符')
                    models = _restore_models(rec['models'])
                    _refine_counts['rf3_pairs_validated_or_reused'] += 1
                    pool = []
                    for mi,out in models:
                        check = _assess_model(seq_aa,out.atom_array,out.summary_confidences,expected_binder_seq=input_seq)
                        _refine_model_audit.append({**pair_base,'model_index':mi,'valid_output':check['valid_output'],
                            'errors':'|'.join(check['errors']),**{key:_metric_csv(v) for key,v in check['metrics'].items()}})
                        _refine_counts['rf3_models'] += 1
                        # SR56_FEEDBACK_BEGIN
                        _feedback_safe(lambda: _feedback_model(rec,mi,out,check,di,si,backbone_file,seq_file,rnd,pi,k,parent_path))
                        # SR56_FEEDBACK_END
                        _refine_counts['rf3_valid_models'] += int(check['valid_output'])
                        if check['valid_output']: pool.append((mi,out,check['metrics']))
                    selected = max(pool,key=lambda t:_rank_key(t[1].summary_confidences,(t[0],))) if pool else None
                    # SR56_FEEDBACK_BEGIN
                    _feedback_safe(lambda: _feedback_pair(rec,selected,di,si,backbone_file,seq_file,rnd,pi,k,parent_path))
                    # SR56_FEEDBACK_END
                    _refine_pair_rows.append({**pair_base,'n_generated':len(models),'n_valid':len(pool),
                        'n_invalid':len(models)-len(pool),'valid_fraction':len(pool)/len(models) if models else '',
                        'best_model_index':selected[0] if selected else '',
                        'status':'valid_prediction_output' if selected else ('no_output' if not models else 'invalid_outputs'),
                        'errors':'' if selected else '|'.join(sorted({r['errors'] for r in _refine_model_audit if all(r[a]==pair_base[a] for a in pair_base)}))})
                    if selected:
                        mi,out,metrics = selected
                        all_candidates.append({'pi':pi,'parent_path':parent_path,'k':k,'si':si,'mi':mi,'out':out,
                            'summary':_report_summary(out.summary_confidences),'metrics':metrics,
                            'backbone':backbone,'mpnn_aa':seq_aa,'backbone_file':backbone_file,'seq_file':seq_file,
                            'model_cif':rec['models'][mi]['cif'],'pair_checkpoint':str(pair_ckpt)})
            finally:
                _release_engines()

            all_candidates.sort(key=lambda c:_rank_key(c['summary'],(c['pi'],c['k'],c['si'],c['mi'])),reverse=True)
            topk = []; seen = set()
            for c in all_candidates:
                source = (c['pi'],c['k'])
                if source in seen: continue
                seen.add(source); topk.append(c)
                if len(topk)==refine_beam_k: break
            beam_kept[di][rnd] = len(topk)

            # SR56_DISPLAY_BEGIN
            _sr56_display_retained.extend({'design_index':di,'round':rnd,'path':f'{_c["parent_path"]}-C{rnd}P{_c["pi"]}K{_c["k"]}S{_c["si"]}M{_c["mi"]}', 'candidate_id':_feedback_id(di,_c['si'],_c['mi'],rnd,_c['pi'],_c['k']), 'sequence':_binder_seq_of(_c['mpnn_aa']), **_c['summary'], **_c['metrics']} for _c in topk)
            # SR56_DISPLAY_END
            trace = [(c['pi'],c['k'],c['si'],c['mi']) for c in topk]
            round_key = f'D{di}_C{rnd}_round'
            # SR56_FEEDBACK_BEGIN
            _feedback_safe(lambda: _feedback.event('START', 'refine_round', key=round_key))
            # SR56_FEEDBACK_END
            saved_round = _stage_record(_REFINE_CKPT,round_key,round_fp)
            # SR56_FEEDBACK_BEGIN
            _feedback_safe(lambda: _feedback.event('REUSED', 'refine_round', key=round_key, **saved_round)) if saved_round is not None else None
            # SR56_FEEDBACK_END
            if saved_round is not None and saved_round['selected']!=[list(t) for t in trace]:
                raise RuntimeError('恢复的 beam 选择与原轮末记录不一致')
            _resume.save_stage(_REFINE_CKPT,round_key,{'cfg':round_fp,'selected':trace,'n_candidates':len(all_candidates),
                'parents':[p for _,_,p in beam],'status':'completed' if topk else 'no_valid_new_output'},
                files=[c['pair_checkpoint'] for c in topk])
            # SR56_FEEDBACK_BEGIN
            _feedback_safe(lambda: _feedback.event('COMMITTED', 'refine_round', key=round_key, design_index=di, round=rnd, selected=trace, n_candidates=len(all_candidates)))
            # SR56_FEEDBACK_END
            _refine_counts['rounds_completed'] += 1
            # SR56_FEEDBACK_BEGIN
            _feedback_safe(lambda: _feedback_beam(di,rnd,topk))
            # SR56_FEEDBACK_END
            if not topk:
                print(f'D{di} C{rnd} 没有有效新输出；保留 baseline/此前最佳结果'); break
            c = topk[0]
            last_out = c['out']; current_aa = last_out.atom_array; last_summary = c['summary']
            last_round,last_k,last_s = rnd,c['k'],c['si']
            path = f'{c["parent_path"]}-C{rnd}P{c["pi"]}K{c["k"]}S{c["si"]}M{c["mi"]}'
            refine_history[di].append((rnd,c['metrics']['binder_pose_rmsd_A'],last_summary['overall_plddt'],last_summary['ptm'],
                c['k'],c['si'],path,c['metrics']['binder_self_rmsd_A'],c['metrics']['target_fragment_rmsd_A']))
            refine_rankings[di][rnd] = last_summary['ranking_score']; refine_iptms[di][rnd] = last_summary['iptm']
            key = _rank_key(last_summary,(rnd,c['pi'],c['k'],c['si'],c['mi']))
            if key>best_key:
                best_key = key
                refine_best_round[di] = (rnd,c['k'],c['si'],current_aa,last_summary,last_out,path)
                refine_best_gen[di] = c['backbone']; refine_best_mpnn[di] = c['mpnn_aa']
                refine_best_model_index[di] = c['mi']
            beam = [(pi,_reset_fragment_target(c['out'].atom_array),
                    f'{c["parent_path"]}-C{rnd}P{c["pi"]}K{c["k"]}S{c["si"]}M{c["mi"]}') for pi,c in enumerate(topk)]
            print(f'D{di} C{rnd} 有效候选 {len(all_candidates)}；后续路径 {len(beam)}；最佳轮 {refine_best_round[di][0]}')

        refine_best[di] = (orig_aa,current_aa,refine_history[di][-1][1],last_summary,last_round,last_k,last_s,last_out)
        _resume.save_stage(_REFINE_CKPT,f'design_{di}',{'cfg':design_fp,'complete':True,
                           'best_round':refine_best_round[di][0],'best_model_index':refine_best_model_index[di],'last_round':last_round,
                           'n_history_rows':len(refine_history[di]),'beam_kept':beam_kept[di]})
finally:
    _release_engines()

with open(REFINE_DIR/'refine_backbone_validity.csv','w',newline='') as f:
    w = _csv.DictWriter(f,fieldnames=['design_index','round','parent_index','path','backbone','valid_output','errors'])
    w.writeheader(); w.writerows(_refine_backbone_audit)
with open(REFINE_DIR/'refine_model_audit.csv','w',newline='') as f:
    w = _csv.DictWriter(f,fieldnames=['design_index','round','parent_index','path','backbone','sequence_index',
                                    'model_index','valid_output','errors',*QC_METRIC_FIELDS])
    w.writeheader(); w.writerows(_refine_model_audit)
with open(REFINE_DIR/'refine_pair_summary.csv','w',newline='') as f:
    w = _csv.DictWriter(f,fieldnames=['design_index','round','parent_index','path','backbone','sequence_index',
                                    'n_generated','n_valid','n_invalid','valid_fraction','best_model_index','status','errors'])
    w.writeheader(); w.writerows(_refine_pair_rows)
_refine_counts['rf3_pair_limit'] = n_top*(1+max(0,refine_rounds-1)*refine_beam_k)*refine_n_backbones*(refine_mpnn_seqs if refine_rf3_validations==0 else min(refine_rf3_validations,refine_mpnn_seqs))
assert _refine_counts['rf3_pairs_validated_or_reused']<=_refine_counts['rf3_pair_limit']
_write_json_atomic(REFINE_DIR/'refine_stage_counts.json',_refine_counts)
shortpep_monitor_update('REFINE_COMPLETE',**_refine_counts)
print('Refine 完整计数:',_refine_counts)
# SR56_FEEDBACK_BEGIN
_feedback_safe(lambda: _feedback.event('END', 'refine', status='COMPLETED'))
# SR56_FEEDBACK_END

# SR56_DISPLAY_BEGIN
_sr56_table('Refine baseline / 每轮代表 / retained 路径', _sr56_refine_rows(), [('design_index', '设计'), ('row_type', '记录类型'), ('round', '轮次'), ('path', '路径'), ('candidate_id', '模型身份'), ('whole_loop_best', '全循环最佳'), ('ranking_score', 'ranking'), ('iptm', 'ipTM'), ('overall_plddt', 'pLDDT'), ('ptm', 'pTM'), ('binder_self_rmsd_A', 'binder self RMSD Å'), ('binder_pose_rmsd_A', 'binder pose RMSD Å'), ('target_fragment_rmsd_A', 'target RMSD Å'), ('sequence', '完整 binder 序列')], REFINE_RANKING_DIR/'refine_path_display.csv', '真正 retained 节点来自原 topk；不会按展示重选 beam。')
# SR56_DISPLAY_END


In [ ]:
# 【固定流程 · 无需修改】展示各轮代表与保留路径。

# SR56_DISPLAY_BEGIN
_sr56_best_display_metrics = {}
# SR56_DISPLAY_END
# ── 原循环总报告、全循环最佳轮 CSV / FASTA / CIF 导出位置 ──
with open(REFINE_RANKING_DIR/'refine_all_report.csv','w',newline='') as f:
    w = _csv.writer(f)
    w.writerow(['design_index','round','path','backbone','sequence_index','ranking_score','iptm','overall_plddt','ptm',
                'binder_pose_rmsd_A','binder_self_rmsd_A','target_fragment_rmsd_A'])
    for di,hist in refine_history.items():
        for rnd,pose,plddt,ptm,k,si,path,self_r,fragment in hist:
            w.writerow([di,rnd,path,k,si,_metric_csv(refine_rankings[di][rnd]),_metric_csv(refine_iptms[di][rnd]),
                        _metric_csv(plddt),_metric_csv(ptm),_metric_csv(pose),_metric_csv(self_r),_metric_csv(fragment)])
_best_sorted = sorted(refine_best_round.items(),key=lambda item:_rank_key(item[1][4],(item[0],item[1][0],item[1][1],item[1][2])),reverse=True)
with open(REFINE_RANKING_DIR/'best_round_summary.csv','w',newline='') as f, open(REFINE_RF3_DIR/'best_round_sequences.fasta','w') as fasta:
    w = _csv.writer(f)
    w.writerow(['rank','design_index','best_round','path','backbone','sequence_index','model_index','ranking_score','iptm',*QC_METRIC_FIELDS])
    for rank,(di,best) in enumerate(_best_sorted,1):
        rnd,k,si,aa,summary,out,path = best
        check = _assess_model(refine_best_mpnn[di],aa,summary,expected_binder_seq=_binder_seq_of(refine_best_mpnn[di]))
        if not check['valid_output']: raise RuntimeError('最佳 refine 导出身份检查失败')

        # SR56_DISPLAY_BEGIN
        _sr56_best_display_metrics[di] = dict(check['metrics'])
        # SR56_DISPLAY_END
        stem = f'best_refine_{pdb_tag}_D{di}_C{rnd}_K{k}_S{si}'
        to_cif_file(refine_best_gen[di],str(REFINE_RF3_DIR/(stem+'_generated.cif')))
        to_cif_file(refine_best_mpnn[di],str(REFINE_RF3_DIR/(stem+'_mpnn.cif')))
        to_cif_file(aa,str(REFINE_RF3_DIR/(stem+'_refolded.cif')))
        to_cif_file(_extract_binder(aa),str(REFINE_RF3_DIR/(stem+'_binder.cif')))
        fasta.write(f'>{stem} path={path} model_index={refine_best_model_index[di]} ranking={_metric_csv(summary["ranking_score"])}\n{_binder_seq_of(aa)}\n')
        w.writerow([rank,di,rnd,path,k,si,refine_best_model_index[di],_metric_csv(summary['ranking_score']),_metric_csv(summary['iptm']),
                    *[_metric_csv(check['metrics'][key]) for key in QC_METRIC_FIELDS]])
print('📁 全循环最佳轮导出:',REFINE_DIR)
# SR56_FEEDBACK_BEGIN
for _feedback_rank, (_feedback_di, _feedback_best) in enumerate(_best_sorted, 1):
    _feedback_safe(lambda: _feedback.event('FINAL_SELECTED', 'refine_rf3', key=_feedback_paths[(_feedback_di, _feedback_best[6])], id=_feedback_paths[(_feedback_di, _feedback_best[6])], design_index=_feedback_di, round=_feedback_best[0], path=_feedback_best[6], formal_rank=_feedback_rank, selection='refine'))
# SR56_FEEDBACK_END

# SR56_DISPLAY_BEGIN
_sr56_table('全循环正式最佳榜 · 完整序列', ({'rank':rank,'design_index':di,'round':best[0],'path':best[6],'sequence_index':best[2],'model_index':refine_best_model_index[di],'sequence':_binder_seq_of(refine_best_mpnn[di]),**best[4],**_sr56_best_display_metrics.get(di,{})} for rank,(di,best) in enumerate(_best_sorted,1)), [('round','最佳轮'),('path','实际路径')]+[('rank', '原榜名次'), ('design_index', '设计'), ('sequence_index', '序列编号'), ('model_index', '模型编号'), ('ranking_score', 'ranking'), ('overall_plddt', 'pLDDT'), ('ptm', 'pTM'), ('iptm', 'ipTM'), ('overall_pae', 'PAE'), ('binder_self_rmsd_A', 'binder self RMSD Å'), ('binder_pose_rmsd_A', 'binder pose RMSD Å'), ('target_fragment_rmsd_A', 'target RMSD Å'), ('sequence', '完整 binder 序列')], REFINE_RANKING_DIR/'refine_best_display.csv', 'baseline 可继续成为最佳；原 best_round_summary.csv 及 FASTA 保持。')
# SR56_DISPLAY_END


In [ ]:
# 【固定流程 · 无需修改】绘制逐轮指标和全循环最佳路径曲线。
# 📈 迭代收敛曲线（每轮binder 姿态 RMSD）— 零依赖 SVG 渲染
# ═══════════════════════════════════════════════════════════════
from IPython.display import HTML

# 收集曲线数据
import html as _sr56_svg_html
curves = []
curve_best = []
max_rmsd = plot_rmsd_scale_min
for rank_idx in range(n_top):
    d_idx = rmsd_report[rank_idx][0]
    short = f"{pdb_tag}_B{d_idx//diffusion_batch_size}_D{d_idx%diffusion_batch_size}"
    _rb = refine_best_round[d_idx]
    _selected_model = refine_best_model_index[d_idx]
    _last_observed = refine_history[d_idx][-1][0] if refine_history[d_idx] else 0
    _disp = f"{short} 最佳 C{_rb[0]} K{_rb[1]} S{_rb[2]} M{_selected_model} {_rb[6]}；最后观察 C{_last_observed}"
    hist = refine_history[d_idx]
    _pairs = [(int(h[0]), _sr56_plot_value(h[1])) for h in hist if _sr56_plot_value(h[1]) is not None]   # 修复：round/value 成对过滤（避免轮次与数值错位）
    _best_point = next(((int(h[0]),_sr56_plot_value(h[1])) for h in hist if h[0]==_rb[0] and h[6]==_rb[6] and _sr56_plot_value(h[1]) is not None), None)
    curve_best.append(_best_point)
    rounds = [p[0] for p in _pairs]
    rmsds  = [p[1] for p in _pairs]
    curves.append((_sr56_svg_html.escape(_disp + (" 🎯" if converged.get(d_idx) else "")), rounds, rmsds))
    if rmsds:
        max_rmsd = max(max_rmsd, max(rmsds) + 0.3)

_W, _H = 860, 420
_PAD_L, _PAD_R, _PAD_T, _PAD_B = 55, 15, 30, 45
_COLORS = ["#0D47A1", "#E65100", "#2E7D32", "#6A1B9A", "#C62828"]
_plot_w = _W - _PAD_L - _PAD_R
_plot_h = _H - _PAD_T - _PAD_B
_max_rnd = max(1, max((max(c[1]) for c in curves if c[1]), default=0))   # 修复：仅第0轮时零除保护

def _px(r, rmsd):
    x = _PAD_L + (r / _max_rnd) * _plot_w
    y = _PAD_T + _plot_h - (rmsd / max_rmsd) * _plot_h
    return x, y

svg_parts = [f'<svg width="{_W}" height="{_H}" xmlns="http://www.w3.org/2000/svg" font-family="Arial,sans-serif">']
# 背景网格 + Y 轴刻度
for g in range(5):
    gy = _PAD_T + _plot_h * (1 - g / 4)
    val = max_rmsd * g / 4
    svg_parts.append(f'<line x1="{_PAD_L}" y1="{gy:.1f}" x2="{_W-_PAD_R}" y2="{gy:.1f}" stroke="#e0e0e0" stroke-width="1"/>')
    svg_parts.append(f'<text x="{_PAD_L-8}" y="{gy+4:.1f}" font-size="11" text-anchor="end" fill="#666">{val:.2f}</text>')
# X 轴刻度
for r in range(_max_rnd + 1):
    rx, _ = _px(r, 0)
    svg_parts.append(f'<line x1="{rx:.1f}" y1="{_PAD_T+_plot_h}" x2="{rx:.1f}" y2="{_PAD_T+_plot_h+4}" stroke="#999" stroke-width="1"/>')
    svg_parts.append(f'<text x="{rx:.1f}" y="{_PAD_T+_plot_h+18}" font-size="11" text-anchor="middle" fill="#666">第{r}轮</text>')
# 各设计曲线
for ci, (label, rounds, rmsds) in enumerate(curves):
    color = _COLORS[ci % len(_COLORS)]
    pts = [_px(r, v) for r, v in zip(rounds, rmsds)]
    poly = " ".join(f"{x:.1f},{y:.1f}" for x, y in pts)
    svg_parts.append(f'<polyline points="{poly}" fill="none" stroke="{color}" stroke-width="2.5"/>')
    for (x, y), v in zip(pts, rmsds):
        svg_parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="4.5" fill="{color}"/>')
        svg_parts.append(f'<text x="{x:.1f}" y="{y-9:.1f}" font-size="10" text-anchor="middle" fill="{color}">{v:.2f}</text>')
    if curve_best[ci] is not None:
        _bx,_by = _px(*curve_best[ci])
        svg_parts.append(f'<circle cx="{_bx:.1f}" cy="{_by:.1f}" r="8" fill="none" stroke="{color}" stroke-width="2.5"><title>实际全循环最佳模型</title></circle>')
    svg_parts.append(f'<text x="{_PAD_L+10}" y="{_PAD_T+18+ci*18}" font-size="10" fill="{color}">● {label}</text>')
svg_parts.append(f'<text x="{_W/2}" y="18" font-size="14" font-weight="bold" text-anchor="middle" fill="#333">Refine 姿态诊断曲线（binder 姿态 RMSD）</text>')
svg_parts.append('</svg>')
display(_collapsible("📈 Refine 姿态诊断曲线（binder 姿态 RMSD）", HTML("".join(svg_parts))))

# ── 导出 SVG 收敛曲线到文件 ──
svg_path = SVG_REFINE_DIR / "rmsd_curves.svg"
with open(svg_path, "w") as f_svg:
    f_svg.write("".join(svg_parts))
print(f"  📈 收敛曲线已导出: {svg_path.resolve()}")

In [ ]:
# 【固定流程 · 无需修改】展示全循环最佳与固定 baseline；提交最终报告与反馈包。
#  全循环最佳轮 3D 对比图（生成骨架 | RF3 重折叠 | 叠加偏差）
# ═══════════════════════════════════════════════════════════════
for rank_idx in range(n_top):
    d_idx = rmsd_report[rank_idx][0]
    short = f"{pdb_tag}_B{d_idx//diffusion_batch_size}_D{d_idx%diffusion_batch_size}"
    _best_r = refine_best_round[d_idx]      # (最优轮C, K, S, rf3_aa, summary, rf3_output, 路径标记)
    _bc, _bk, _bs_r = _best_r[0], _best_r[1], _best_r[2]
    _best_display_mi = refine_best_model_index[d_idx]
    _path_m = _best_r[6] if len(_best_r) > 6 else ""
    short_cs = f"{short}_C{_bc}_{_path_m}" if _path_m else (f"{short}_C{_bc}_K{_bk}_S{_bs_r}" if _bk >= 0 else f"{short}（baseline）")
    short_cs += f"_M{_best_display_mi}"
    orig_aa_f = refine_best_gen[d_idx]   # 与全循环最佳 RF3 输出对应的实际生成骨架
    rf3_aa_f  = _best_r[3]                # 全循环最优轮 RF3 重折叠
    s_f       = _best_r[4]                # 最优轮 summary
    _rmsd_hist = dict((h[0], h[1]) for h in refine_history[d_idx])
    rmsd_f    = _rmsd_hist.get(_bc, float('nan'))   # 最优轮binder 姿态 RMSD
    print(f"\n{'═'*60}")
    _ip27 = s_f.get('iptm')
    _ip27s = _sr56_fmt(_ip27)
    print(f"  🔬 {short_cs}  最优轮RMSD={_sr56_fmt(rmsd_f)}Å  ranking={_sr56_fmt(s_f.get('ranking_score'))}  ipTM={_ip27s}")
    print(f"{'═'*60}")
    _sr56_three_panel('全循环最佳与主阶段 baseline · '+short_cs, [{
        'design_index':d_idx,'label':short_cs,'sequence':_binder_seq_of(refine_best_mpnn[d_idx]),
        'summary':s_f,'metrics':_sr56_best_display_metrics.get(d_idx,{}),
        'rfd3':orig_aa_f,'mpnn':refine_best_mpnn[d_idx],'rf3':rf3_aa_f,
        'rf3_confidences':getattr(_best_r[5], 'confidences', {}) if _best_r[5] is not None else getattr(design_cand[d_idx]['rf3_out'],'confidences',{})}])
# SR56_FEEDBACK_BEGIN
_feedback_safe(lambda: _feedback.event('END', 'export', status='COMPLETED'))
# SR56_FEEDBACK_END
# SR56_FEEDBACK_BEGIN
_feedback_safe(lambda: _feedback.finish(status='COMPLETED'))
# SR56_FEEDBACK_END


---
### 运行与恢复

默认已填 LARGE256 与三轮 refine；不自动安装依赖或下载权重。内置输入解包到本分支 prep/input_sources；外部输入与运行输出目录分离。缓存记录目标、参数、引擎来源、权重与工件散列；旧 proxy、folded、不同配置断点拒绝复用。运行错误停止并保留已完成阶段，不自动降低预算。

原始排名、模型数、有效数、无效数据原因、接触位置和 RMSD 均可追溯。没有有效输出时产生空表和明确状态，不补入历史候选。


目录：main/refine 保存计算结果；rankings、reports、svg 按用途分区；monitor 保存观察记录；checkpoints 保存身份、冻结 notebook 和断点。新参数新运行；同身份续跑必须指定原目录。反馈快照位于本次运行的 feedback，不覆盖已有 ZIP。


---
### 验收边界

布局、语法、CPU 假引擎循环及恢复检查覆盖文件和控制流程；真实 GPU 推理与显存须单独核验。计算预测仅围绕孤立片段的 binder 设计，不执行折叠态反筛。
